### `add_backgrounds.py`

add_backgrounds.py

In [ ]:
"""
add_backgrounds.py

Inject BACKGROUND / NEGATIVE images (empty scenes with NO rat) into the
merged dataset so the detector learns what "nothing here" looks like and
stops raising false alarms on shadows, food, reflections, etc.

Each background is written into all three places your pipeline uses:
  1. datasets/merged_dataset/images/    -> the .jpg image
  2. datasets/merged_dataset/labels/    -> an EMPTY .txt file (YOLO background)
  3. datasets/merged_dataset/ei_upload/ -> the .jpg + an empty-box JSON
                                           {"version": 1, "boundingBoxes": []}

HOW TO USE
----------
1. Put your empty-scene images (captured from the REAL camera at the feeding
   station: empty, food-but-no-rat, day + night/IR, shadows) into a folder,
   e.g.  backgrounds_source/
2. Run:  python add_backgrounds.py
3. Re-upload the ei_upload folder to Edge Impulse and retrain.

Notes
-----
* Background images are named  bg_000001.jpg, bg_000002.jpg, ...  so they are
  easy to spot and never collide with your numbered positives.
* An EMPTY .txt label is the correct YOLO way to mark a background image.
  Your dataset_check.py will list these as "empty_label" -- that is EXPECTED
  for backgrounds and is NOT an error.
* Aim for roughly 10-20% backgrounds relative to your positive images.

Requires: pillow  (pip install pillow)
"""

import json
import sys
from pathlib import Path

from PIL import Image, UnidentifiedImageError

# ----------------------------- configuration ------------------------------
SOURCE_DIR = "backgrounds_source"          # put your empty-scene images here
MERGED_DIR = "datasets/merged_dataset"
PREFIX = "bg_"                             # naming prefix for backgrounds
JPEG_QUALITY = 90
RESIZE_MAX_DIM = None                      # e.g. 1024 to cap size; None = keep
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
# --------------------------------------------------------------------------


def next_start_index(images_dir: Path) -> int:
    """Continue numbering after any bg_ files that already exist."""
    max_idx = 0
    for p in images_dir.glob(f"{PREFIX}*.jpg"):
        digits = p.stem[len(PREFIX):]
        if digits.isdigit():
            max_idx = max(max_idx, int(digits))
    return max_idx + 1


def load_rgb(path: Path):
    """Open an image, verify it is not corrupted, return an RGB copy."""
    try:
        with Image.open(path) as probe:
            probe.verify()                     # detect corruption
        im = Image.open(path).convert("RGB")   # reopen after verify
    except (UnidentifiedImageError, OSError, SyntaxError, ValueError) as e:
        return None, f"unreadable/corrupted: {e}"
    if RESIZE_MAX_DIM:
        im.thumbnail((RESIZE_MAX_DIM, RESIZE_MAX_DIM), Image.LANCZOS)
    return im, None


def main():
    src = Path(SOURCE_DIR)
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"
    ei_dir = Path(MERGED_DIR) / "ei_upload"

    if not src.is_dir():
        print(f"ERROR: source folder not found: {src.resolve()}")
        print("Create it, drop your empty-scene images inside, then re-run.")
        sys.exit(1)

    for d in (images_dir, labels_dir, ei_dir):
        d.mkdir(parents=True, exist_ok=True)

    sources = sorted(p for p in src.iterdir()
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    if not sources:
        print(f"No images found in {src.resolve()} "
              f"(looked for {sorted(IMAGE_EXTS)}).")
        sys.exit(0)

    idx = next_start_index(images_dir)
    added = 0
    skipped = 0

    print(f"Found {len(sources)} candidate background images. Injecting...\n")
    for p in sources:
        im, err = load_rgb(p)
        if err:
            skipped += 1
            print(f"  SKIP {p.name}: {err}")
            continue

        name = f"{PREFIX}{idx:06d}"
        idx += 1

        # 1) YOLO image
        im.save(images_dir / f"{name}.jpg", "JPEG", quality=JPEG_QUALITY)
        # 2) YOLO empty label (marks the image as background)
        (labels_dir / f"{name}.txt").write_text("")
        # 3) Edge Impulse image + empty-box JSON
        im.save(ei_dir / f"{name}.jpg", "JPEG", quality=JPEG_QUALITY)
        (ei_dir / f"{name}.json").write_text(
            json.dumps({"version": 1, "boundingBoxes": []})
        )
        added += 1

    # summary + ratio guidance
    positives = len([q for q in images_dir.glob("*.jpg")
                     if not q.stem.startswith(PREFIX)])
    backgrounds = len(list(images_dir.glob(f"{PREFIX}*.jpg")))
    ratio = (backgrounds / positives * 100) if positives else 0

    print("\nDone.")
    print(f"  Backgrounds added this run: {added}")
    print(f"  Skipped (bad images):       {skipped}")
    print(f"  Total positives:            {positives}")
    print(f"  Total backgrounds:          {backgrounds} "
          f"({ratio:.1f}% of positives)")
    if ratio < 10:
        print("  TIP: aim for ~10-20% backgrounds; add more empty-scene frames.")
    elif ratio > 30:
        print("  NOTE: >30% backgrounds can start to hurt recall.")


if __name__ == "__main__":
    main()


### `analyze_redundancy.py`

One-off dataset redundancy analysis.

In [ ]:
"""One-off dataset redundancy analysis.

Hashes every image with a 64-bit perceptual hash (pHash), clusters near-
duplicates at several Hamming-distance thresholds, prints how much of the
dataset is redundant, and saves a montage of the biggest duplicate clusters.
"""
import time
from pathlib import Path

import numpy as np
from PIL import Image
import imagehash

IMAGES_DIR = Path("datasets/merged_dataset/images")
THRESHOLDS = [2, 5, 8, 10]      # Hamming distance on 64-bit pHash
MONTAGE_THR = 8                 # threshold used to pick clusters to show
MONTAGE_CLUSTERS = 6            # number of biggest clusters to display
MONTAGE_PER = 8                 # thumbnails per cluster (row)
THUMB = 128
OUT_PNG = Path("datasets/redundancy_clusters.png")


def main():
    paths = sorted(IMAGES_DIR.glob("*.jpg"))
    if not paths:
        print(f"No images in {IMAGES_DIR.resolve()}")
        return
    print(f"Found {len(paths)} images. Hashing (pHash)...")

    packed_list, kept = [], []
    t0 = time.time()
    for i, p in enumerate(paths):
        try:
            h = imagehash.phash(Image.open(p))          # 8x8 bool
            packed_list.append(np.packbits(h.hash.flatten()))
            kept.append(p)
        except Exception as e:  # noqa: BLE001 - skip unreadable files
            print(f"  skip {p.name}: {e}")
        if (i + 1) % 2000 == 0:
            print(f"  hashed {i + 1}/{len(paths)}  ({time.time() - t0:.0f}s)")

    packed = np.array(packed_list, dtype=np.uint8)       # (N, 8)
    N = len(packed)
    print(f"Hashed {N} images in {time.time() - t0:.0f}s\n")

    popcount = np.array([bin(x).count("1") for x in range(256)], dtype=np.uint16)

    parent = {thr: np.arange(N) for thr in THRESHOLDS}

    def find(par, x):
        while par[x] != x:
            par[x] = par[par[x]]
            x = par[x]
        return x

    def union(par, a, b):
        ra, rb = find(par, a), find(par, b)
        if ra != rb:
            par[rb] = ra

    print("Clustering near-duplicates...")
    t0 = time.time()
    for i in range(N):
        dist = popcount[packed ^ packed[i]].sum(axis=1)   # (N,)
        dist[: i + 1] = 255                               # only j > i
        for thr in THRESHOLDS:
            par = parent[thr]
            for j in np.where(dist <= thr)[0]:
                union(par, i, int(j))
        if (i + 1) % 2000 == 0:
            print(f"  clustered {i + 1}/{N}  ({time.time() - t0:.0f}s)")

    print("\nRedundancy by similarity threshold (0 = identical, higher = looser):")
    print(f"{'thr':>4} {'distinct':>9} {'redundant':>10} {'%redundant':>11}")
    cluster_data = {}
    for thr in THRESHOLDS:
        par = parent[thr]
        roots = np.array([find(par, i) for i in range(N)])
        uniq, counts = np.unique(roots, return_counts=True)
        redundant = N - len(uniq)
        print(f"{thr:>4} {len(uniq):>9} {redundant:>10} {100 * redundant / N:>10.1f}%")
        cluster_data[thr] = (roots, uniq, counts)

    roots, uniq, counts = cluster_data[MONTAGE_THR]
    order = np.argsort(-counts)
    print(f"\nBiggest near-duplicate clusters at thr={MONTAGE_THR}:")
    big = []
    for k in order[:MONTAGE_CLUSTERS]:
        members = [kept[idx] for idx in np.where(roots == uniq[k])[0]]
        big.append(members)
        print(f"  {counts[k]:>4} near-identical frames  (e.g. {members[0].name})")

    singletons = int((counts == 1).sum())
    print(f"\nUnique one-off images (no near-duplicate) at thr={MONTAGE_THR}: {singletons}")

    rows = len(big)
    canvas = Image.new("RGB", (MONTAGE_PER * THUMB, rows * THUMB), (20, 20, 20))
    for r, members in enumerate(big):
        for c in range(min(MONTAGE_PER, len(members))):
            try:
                im = Image.open(members[c]).convert("RGB").resize((THUMB, THUMB))
                canvas.paste(im, (c * THUMB, r * THUMB))
            except Exception:  # noqa: BLE001
                pass
    OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(OUT_PNG)
    print(f"\nSaved montage -> {OUT_PNG.resolve()}")
    print("Each row = one cluster of near-identical frames (redundant data).")


if __name__ == "__main__":
    main()


### `converting_labels_to_json.py`

Create a new "third" folder for Edge Impulse

In [ ]:
import json
import sys
import shutil
from pathlib import Path
from PIL import Image

DATASET_DIR = "datasets/merged_dataset"
# Create a new "third" folder for Edge Impulse
OUTPUT_DIR = Path(DATASET_DIR) / "ei_upload"

def create_ei_metadata():
    images_dir = Path(DATASET_DIR) / "images"
    labels_dir = Path(DATASET_DIR) / "labels"
    
    # Create the output folder if it doesn't exist
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    valid_extensions = ['.jpg', '.jpeg', '.png']
    all_images = [p for p in images_dir.iterdir() if p.suffix.lower() in valid_extensions]
    total_images = len(all_images)
    
    if total_images == 0:
        print("No images found in the directory!")
        return

    print(f"Found {total_images} images. Copying images and creating JSON in '{OUTPUT_DIR.name}'...\n")
    
    processed_count = 0
    skipped_count = 0

    for img_path in all_images:
        label_path = labels_dir / f"{img_path.stem}.txt"
        
        # Define where the new files will go
        out_img_path = OUTPUT_DIR / img_path.name
        out_json_path = OUTPUT_DIR / f"{img_path.stem}.json"

        if not label_path.exists():
            skipped_count += 1
            processed_count += 1
            sys.stdout.write(f"\rProgress: {processed_count}/{total_images} processed (Skipped: {skipped_count})")
            sys.stdout.flush()
            continue

        try:
            # 1. Copy the original image to the new folder
            shutil.copy(img_path, out_img_path)
            
            # 2. Read the original image to get dimensions
            with Image.open(img_path) as img:
                img_w, img_h = img.size
            
            bboxes = []
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    
                    _, x_c, y_c, w, h = map(float, parts)
                    
                    # Convert YOLO to Edge Impulse pixel coordinates
                    x_min = (x_c - w / 2) * img_w
                    y_min = (y_c - h / 2) * img_h
                    width = w * img_w
                    height = h * img_h
                    
                    bboxes.append({
                        "label": "rat",
                        "x": int(x_min),
                        "y": int(y_min),
                        "width": int(width),
                        "height": int(height)
                    })
            
            # 3. Save the new JSON file to the new folder
            metadata = {
                "version": 1,
                "boundingBoxes": bboxes
            }
            
            with open(out_json_path, 'w') as f:
                json.dump(metadata, f)
                
        except Exception as e:
            print(f"\nError processing {img_path.name}: {e}")
            skipped_count += 1

        processed_count += 1
        sys.stdout.write(f"\rProgress: {processed_count}/{total_images} processed (Skipped: {skipped_count})")
        sys.stdout.flush()

    print(f"\n\nDone! Your Edge Impulse data is ready in: {OUTPUT_DIR}")

if __name__ == "__main__":
    create_ei_metadata()

### `curate_and_augment.py`

curate_and_augment.py

In [ ]:
"""
curate_and_augment.py

Builds a smaller, higher-quality training set that looks more like the real
ESP32-CAM feed, then packages it in Edge Impulse upload format.

Two problems it addresses:
  1. You have ~11,783 images -> too many/slow for FOMO, and many are near-dups
     of the same scene (which inflates validation and wastes training budget).
  2. Your images are color / bright / high-res, but the deployment camera is
     grayscale-ish, low-res and often dark (night/IR). This "domain gap" is a
     big reason the unseen test set scored lower than validation.

What it does:
  * CURATE: groups near-identical scenes with a perceptual hash and caps how
    many frames each scene contributes, then selects up to TARGET_COUNT images
    (optionally biased toward the grayscale-looking ones).
  * AUGMENT (domain randomization): for each selected image it emits the
    original plus AUG_PER_IMAGE variants that simulate the ESP32-CAM look
    (grayscale, darken/gamma, low-res pixelation, sensor noise, optional
    horizontal flip). Bounding boxes are carried over and flipped correctly.
  * EXPORT: writes <stem>.jpg + <stem>.json (Edge Impulse bounding-box format)
    into OUTPUT_DIR, ready to drag-and-drop into Edge Impulse Studio.

Requires: pillow, numpy, imagehash   (pip install pillow numpy imagehash)
"""

import json
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageEnhance
import imagehash

# ----------------------------- configuration ------------------------------
IMAGES_DIR = Path("datasets/merged_dataset/images")
LABELS_DIR = Path("datasets/merged_dataset/labels")
OUTPUT_DIR = Path("datasets/ei_curated")
LABEL_NAME = "rat"

TARGET_COUNT = 1500        # number of source images to curate (before augmenting)
MAX_PER_SCENE = 3          # cap near-duplicate frames from the same scene
PHASH_THRESHOLD = 8        # lower = stricter grouping of "same scene"
FAVOR_GRAYSCALE = True     # prefer images already close to the IR/grayscale look

# drop frames that are unrealistic for a single fixed camera
FILTER_COLLAGES = True     # skip stitched/collage frames (interior divider gutters)
MIN_ASPECT = 0.5           # skip very tall images (width/height below this)
MAX_ASPECT = 2.0           # skip very wide images (width/height above this)
COLLAGE_GUTTER_STD = 7.0   # a near-uniform interior line (low row/col std) = candidate
COLLAGE_SIDE_TEXTURE = 22.0  # ...only a real divider if textured content is on BOTH sides

AUG_PER_IMAGE = 2          # augmented copies per curated image (0 = curate only)
TRAIN_RATIO = 0.8          # source-level split; a source + its augs stay together
RANDOM_SEED = 42
JPEG_QUALITY = 90

# per-augmentation probabilities (each variant samples a random combination)
P_GRAYSCALE = 0.7
P_DARKEN = 0.6
P_LOWRES = 0.7
P_NOISE = 0.5
P_HFLIP = 0.5
# --------------------------------------------------------------------------


def read_yolo_boxes(txt_path: Path):
    """Return a list of (cls, xc, yc, w, h) normalized floats."""
    boxes = []
    if not txt_path.exists():
        return boxes
    for line in txt_path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        try:
            cls = int(float(parts[0]))
            xc, yc, w, h = (float(v) for v in parts[1:])
        except ValueError:
            continue
        boxes.append((cls, xc, yc, w, h))
    return boxes


def yolo_to_ei(boxes, img_w, img_h, flipped=False):
    """Convert normalized YOLO boxes to Edge Impulse absolute-pixel boxes."""
    ei_boxes = []
    for _, xc, yc, w, h in boxes:
        if flipped:
            xc = 1.0 - xc
        x = (xc - w / 2.0) * img_w
        y = (yc - h / 2.0) * img_h
        bw = w * img_w
        bh = h * img_h
        # clamp to image bounds
        x = max(0, min(img_w - 1, x))
        y = max(0, min(img_h - 1, y))
        bw = max(1, min(img_w - x, bw))
        bh = max(1, min(img_h - y, bh))
        ei_boxes.append({
            "label": LABEL_NAME,
            "x": int(round(x)),
            "y": int(round(y)),
            "width": int(round(bw)),
            "height": int(round(bh)),
        })
    return ei_boxes


def grayscale_score(im: Image.Image) -> float:
    """Mean per-pixel channel spread. Lower = the image is already grayscale."""
    small = im.convert("RGB").resize((32, 32))
    arr = np.asarray(small).astype(np.int16)
    diff = (np.abs(arr[:, :, 0] - arr[:, :, 1]) +
            np.abs(arr[:, :, 0] - arr[:, :, 2]) +
            np.abs(arr[:, :, 1] - arr[:, :, 2]))
    return float(diff.mean())


def looks_like_collage(im: Image.Image) -> bool:
    """Flag stitched/collage frames by detecting an interior 'gutter': a
    near-uniform line that has textured content on BOTH sides (a divider between
    two panels). A plain sky or white studio background is uniform too, but its
    neighbouring lines are also uniform, so it is NOT flagged."""
    g = np.asarray(im.convert("L").resize((160, 160))).astype(np.float32)
    n = g.shape[0]
    lo, hi = int(n * 0.15), int(n * 0.85)

    def has_divider(line_std):
        for i in range(lo, hi):
            if line_std[i] < COLLAGE_GUTTER_STD:
                above = line_std[max(0, i - 12):max(1, i - 3)].mean()
                below = line_std[i + 3:i + 12].mean()
                if above > COLLAGE_SIDE_TEXTURE and below > COLLAGE_SIDE_TEXTURE:
                    return True
        return False

    return has_divider(g.std(axis=1)) or has_divider(g.std(axis=0))


def curate(paths):
    """Group by perceptual hash, cap per scene, optionally bias to grayscale."""
    print("Curating: hashing images and grouping scenes (this may take a bit)...")
    items = []
    skipped_collage = 0
    skipped_aspect = 0
    for p in paths:
        try:
            with Image.open(p) as im:
                im = im.convert("RGB")
                w, h = im.size
                ar = w / h if h else 0
                if ar < MIN_ASPECT or ar > MAX_ASPECT:
                    skipped_aspect += 1
                    continue
                if FILTER_COLLAGES and looks_like_collage(im):
                    skipped_collage += 1
                    continue
                ph = imagehash.phash(im)
                score = grayscale_score(im) if FAVOR_GRAYSCALE else 0.0
            items.append((p, ph, score))
        except Exception as e:  # noqa: BLE001 - report and skip bad files
            print(f"  skip {p.name}: {e}")

    print(f"  filtered out {skipped_collage} collage-like and "
          f"{skipped_aspect} extreme-aspect images")
    if not items:
        return []

    # sort by hash so near-identical scenes sit next to each other
    items.sort(key=lambda t: str(t[1]))
    groups = []
    cur = [items[0]]
    for i in range(1, len(items)):
        if (items[i][1] - items[i - 1][1]) < PHASH_THRESHOLD:
            cur.append(items[i])
        else:
            groups.append(cur)
            cur = [items[i]]
    groups.append(cur)
    print(f"  found {len(groups)} scene groups from {len(items)} images")

    # within each scene keep up to MAX_PER_SCENE (most grayscale first)
    selected = []
    for g in groups:
        g_sorted = sorted(g, key=lambda t: t[2])
        selected.extend(g_sorted[:MAX_PER_SCENE])

    random.shuffle(selected)
    if FAVOR_GRAYSCALE:
        selected.sort(key=lambda t: t[2])   # most grayscale first
    selected = selected[:TARGET_COUNT]
    print(f"  selected {len(selected)} images "
          f"(cap {MAX_PER_SCENE}/scene, target {TARGET_COUNT})")
    return [p for p, _, _ in selected]


def augment(im: Image.Image):
    """Apply a random domain-randomization combo. Returns (image, flipped)."""
    flipped = False
    out = im.convert("RGB")

    if random.random() < P_HFLIP:
        out = out.transpose(Image.FLIP_LEFT_RIGHT)
        flipped = True

    if random.random() < P_GRAYSCALE:
        out = out.convert("L").convert("RGB")        # keep 3 channels

    if random.random() < P_DARKEN:
        factor = random.uniform(0.35, 0.8)           # simulate night
        out = ImageEnhance.Brightness(out).enhance(factor)

    if random.random() < P_LOWRES:
        w, h = out.size
        scale = random.uniform(0.15, 0.4)            # simulate ESP32-CAM low-res
        small = out.resize((max(8, int(w * scale)), max(8, int(h * scale))),
                           Image.BILINEAR)
        out = small.resize((w, h), Image.BILINEAR)

    if random.random() < P_NOISE:
        arr = np.asarray(out).astype(np.int16)
        noise = np.random.normal(0, random.uniform(4, 14), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)

    return out, flipped


def save_variant(out_img, boxes_yolo, out_dir, stem, flipped, category):
    """Save one image and return its Edge Impulse label entry."""
    w, h = out_img.size
    out_img.save(out_dir / f"{stem}.jpg", "JPEG", quality=JPEG_QUALITY)
    return {
        "path": f"{stem}.jpg",
        "category": category,
        "boundingBoxes": yolo_to_ei(boxes_yolo, w, h, flipped=flipped),
    }


def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    if not IMAGES_DIR.is_dir():
        print(f"ERROR: {IMAGES_DIR.resolve()} not found")
        return

    all_imgs = sorted(p for p in IMAGES_DIR.glob("*.jpg")
                      if not p.stem.startswith("bg_"))
    if not all_imgs:
        print("No positive images found.")
        return

    chosen = curate(all_imgs)
    if not chosen:
        print("Nothing selected; check your dataset paths.")
        return

    # Split at the SOURCE-image level so an image and all of its augmentations
    # land in the same set -> no train/test leakage.
    random.shuffle(chosen)
    n_train = int(len(chosen) * TRAIN_RATIO)
    splits = [
        (chosen[:n_train], OUTPUT_DIR / "training", "training"),
        (chosen[n_train:], OUTPUT_DIR / "testing", "testing"),
    ]

    print("\nAugmenting + exporting to Edge Impulse object detection format...")
    grand_total = 0
    for srcs, out_dir, category in splits:
        out_dir.mkdir(parents=True, exist_ok=True)
        entries = []
        idx = 1
        for p in srcs:
            boxes = read_yolo_boxes(LABELS_DIR / f"{p.stem}.txt")
            try:
                base = Image.open(p).convert("RGB")
            except Exception as e:  # noqa: BLE001 - report and skip bad files
                print(f"  skip {p.name}: {e}")
                continue

            entries.append(save_variant(base, boxes, out_dir,
                                        f"cur_{idx:06d}", False, category))
            idx += 1
            for _ in range(AUG_PER_IMAGE):
                aug_img, flipped = augment(base)
                entries.append(save_variant(aug_img, boxes, out_dir,
                                            f"cur_{idx:06d}", flipped, category))
                idx += 1

        labels_path = out_dir / "bounding_boxes.labels"
        # CLI-uploader format: dict keyed by filename + a "type" field.
        # (The Studio browser uploader silently ignores this file; the
        #  files-array export format is rejected by the CLI as "Invalid type".)
        bboxes = {e["path"]: e["boundingBoxes"] for e in entries}
        labels_path.write_text(json.dumps(
            {"version": 1, "type": "bounding-box-labels",
             "boundingBoxes": bboxes}))
        grand_total += len(entries)
        print(f"  {category}: {len(entries)} images -> {labels_path}")

    print(f"\nDone. Wrote {grand_total} images to {OUTPUT_DIR.resolve()}")
    print("UPLOAD (use the Edge Impulse CLI, NOT the browser uploader):")
    print("  1. Dashboard > Project info: labeling method =")
    print("     'Bounding boxes (object detection)'.")
    print("  2. npm install -g edge-impulse-cli   (needs Node.js)")
    print(f"  3. edge-impulse-uploader --directory {OUTPUT_DIR}")
    print("     (reads training/ + testing/ bounding_boxes.labels, splits by")
    print("      folder name, attaches the rat boxes). Verify a sample shows a")
    print("      box and the labeling queue is 0.")
    print("Tip: also turn ON training augmentation in the FOMO block.")


if __name__ == "__main__":
    main()


### `dataset_check.py`

check_dataset_issues.py

In [ ]:
"""
check_dataset_issues.py

Scans merged_dataset for three types of problems:
  1. Empty label files      -> 0 bytes, or a .txt with no actual annotation lines
  2. Malformed label info   -> lines that don't have exactly 5 values
                                (class x_center y_center width height),
                                or contain non-numeric values
  3. Corrupted images       -> images that fail to open/decode

Nothing is moved or deleted. Results are printed to the console and written
to a CSV report so you can review and decide what to do with each file.

Requires: pillow  (pip install pillow)

Usage:
    python check_dataset_issues.py
"""

import csv
import sys
from pathlib import Path

from PIL import Image, UnidentifiedImageError

MERGED_DIR = "datasets/merged_dataset"
REPORT_PATH = "dataset_issues_report.csv"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def is_empty_label(label_path: Path) -> bool:
    if label_path.stat().st_size == 0:
        return True
    return not label_path.read_text().strip()


def get_malformed_lines(label_path: Path) -> list[str]:
    """Return descriptions of any line that doesn't look like a valid
    'class x_center y_center width height' annotation."""
    problems = []
    lines = [l for l in label_path.read_text().splitlines() if l.strip()]

    for line_no, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            problems.append(f"line {line_no}: expected 5 values, got {len(parts)} -> '{line}'")
            continue
        try:
            cls_id = int(parts[0])
            x, y, w, h = (float(v) for v in parts[1:])
        except ValueError:
            problems.append(f"line {line_no}: non-numeric value -> '{line}'")
            continue
        if cls_id < 0:
            problems.append(f"line {line_no}: negative class id ({cls_id})")
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            problems.append(f"line {line_no}: value(s) out of expected [0,1] range -> '{line}'")

    return problems


def check_image_corrupted(img_path: Path) -> str | None:
    try:
        with Image.open(img_path) as im:
            im.verify()
        with Image.open(img_path) as im:
            im.load()
    except (UnidentifiedImageError, OSError, SyntaxError, ValueError) as e:
        return f"corrupted image: {e}"
    return None


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    if not images_dir.is_dir():
        print(f"ERROR: images folder not found: {images_dir.resolve()}")
        sys.exit(1)
    if not labels_dir.is_dir():
        print(f"ERROR: labels folder not found: {labels_dir.resolve()}")
        sys.exit(1)

    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(image_paths)
    print(f"Checking {total} images + their labels...\n")

    rows = []
    empty_label_count = 0
    malformed_count = 0
    corrupted_count = 0
    missing_label_count = 0

    for i, img_path in enumerate(image_paths, start=1):
        # --- corrupted image check ---
        corrupt_msg = check_image_corrupted(img_path)
        if corrupt_msg:
            corrupted_count += 1
            rows.append([img_path.name, "corrupted_image", corrupt_msg])

        # --- label checks ---
        label_path = labels_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            missing_label_count += 1
            rows.append([img_path.name, "missing_label", "no matching label file"])
        elif is_empty_label(label_path):
            empty_label_count += 1
            rows.append([img_path.name, "empty_label", "label file has no annotation lines"])
        else:
            malformed = get_malformed_lines(label_path)
            for problem in malformed:
                malformed_count += 1
                rows.append([img_path.name, "malformed_label", problem])

        pct = i / total * 100 if total else 100
        sys.stdout.write(f"\r  checked {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

    print()

    with open(REPORT_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image", "issue_type", "details"])
        writer.writerows(rows)

    print("\nDone.")
    print(f"  Corrupted images:        {corrupted_count}")
    print(f"  Empty label files:       {empty_label_count}")
    print(f"  Malformed label lines:   {malformed_count}")
    print(f"  Missing label files:     {missing_label_count}")
    print(f"  Full report written to: {Path(REPORT_PATH).resolve()}")


if __name__ == "__main__":
    main()

### `dedupe_by_label_area.py`

dedupe_by_label_area_ssim.py

In [ ]:
"""
dedupe_by_label_area_ssim.py

Compares the CROPPED LABELED REGION of each image (not the whole picture)
to find duplicates -- catches cases where the same subject was annotated
in two different images with different backgrounds/crops around it. Uses
the same fast two-stage pipeline as dedupe_images_ssim.py so it stays fast
on large datasets:

    Load images + crop labeled boxes
       |
    Compute pHash of each crop
       |
    Group images with similar crop-hashes (loose threshold, candidate filter)
       |
    Run SSIM only inside each group (strict, final confirmation)

HOW A MATCH IS DECIDED (two things must be true):
  1. Same number of boxes, and each box's normalized AREA (w*h) matches
     within AREA_TOLERANCE -- cheap filter, checked first.
  2. Each box's cropped CONTENT matches via SSIM at or above CONTENT_THRESHOLD.

SAFETY: confirmed duplicates (image + label) are MOVED to a review folder,
nothing is deleted.

NOTE: this version processes every candidate group with SSIM confirmation,
no matter how large -- nothing is skipped. Large groups will just take
longer to confirm.

Usage:
    pip install pillow numpy scikit-image imagehash --break-system-packages
    python dedupe_by_label_area_ssim.py
"""

import shutil
import sys
from pathlib import Path

import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import imagehash

MERGED_DIR = "datasets/merged_dataset"
REVIEW_DIR = "datasets/duplicates_by_label_review_ssim"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# How close two normalized box areas (w*h, both already 0-1) need to be to
# even bother comparing pixel content. 0.02 = within 2% relative difference.
AREA_TOLERANCE = 0.02

# Stage 1 (pHash on the crop) -- loose candidate filter.
HASH_SIZE = 8  # 8x8 = 64-bit hash
HASH_GROUP_THRESHOLD = 6

# Stage 2 (SSIM on the crop) -- strict final confirmation (-1 to 1, 1.0 =
# structurally identical).
CONTENT_THRESHOLD = 0.99

# Crops are resized to this fixed size for both the pHash and the SSIM
# comparison, since box crops naturally come out at different pixel sizes.
COMPARE_SIZE = (128, 128)


def parse_boxes(label_path: Path):
    """Return list of (class_id, x_center, y_center, w, h), all normalized
    0-1, from a YOLO .txt label file."""
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        class_id, xc, yc, w, h = parts[:5]
        boxes.append((int(class_id), float(xc), float(yc), float(w), float(h)))
    return boxes


def crop_box(im: Image.Image, xc, yc, w, h):
    """Crop a normalized YOLO box out of a PIL image. Computes pixel
    width/height first, then derives left/top from the center and that
    fixed size (rather than rounding all four edges independently) --
    otherwise two boxes with identical normalized w/h can end up 1px
    different in size after rounding, throwing off the comparison."""
    img_w, img_h = im.size
    box_w = round(w * img_w)
    box_h = round(h * img_h)
    center_x = round(xc * img_w)
    center_y = round(yc * img_h)

    left = max(0, center_x - box_w // 2)
    top = max(0, center_y - box_h // 2)
    right = min(img_w, left + box_w)
    bottom = min(img_h, top + box_h)

    if right <= left or bottom <= top:
        return None
    return im.crop((left, top, right, bottom))


def build_box_records(images_dir: Path, labels_dir: Path):
    """
    For every image that has a label file, crop out each box and compute
    BOTH its pHash (stage 1) and its resized grayscale array (stage 2).
    Returns {img_path: [(area, hash_int, gray_array), ...]} sorted by area
    descending per image.
    """
    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(image_paths)
    print(f"Cropping labeled regions for {total} images...\n")

    records = {}
    for i, img_path in enumerate(image_paths, start=1):
        label_path = labels_dir / f"{img_path.stem}.txt"
        boxes = parse_boxes(label_path)
        if not boxes:
            continue

        try:
            with Image.open(img_path) as im:
                im = im.convert("RGB")
                box_records = []
                for (_cls, xc, yc, w, h) in boxes:
                    crop = crop_box(im, xc, yc, w, h)
                    if crop is None:
                        continue
                    area = w * h
                    h_hash = int(str(imagehash.phash(crop, hash_size=HASH_SIZE)), 16)
                    gray = crop.convert("L").resize(COMPARE_SIZE, Image.LANCZOS)
                    gray_array = np.array(gray, dtype=np.uint8)
                    box_records.append((area, h_hash, gray_array))
                box_records.sort(key=lambda r: r[0], reverse=True)
                if box_records:
                    records[img_path] = box_records
        except Exception as e:
            print(f"\n[WARNING] Could not process {img_path.name}: {e}")

        pct = i / total * 100 if total else 100
        sys.stdout.write(f"\r  processed {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line
    return records


def group_by_hash(records: dict, threshold: int):
    """
    Stage 1: cheap candidate filter, using the LARGEST box's pHash per
    image (the main subject) as a representative signature. Vectorized
    with numpy in row blocks. Deliberately loose (high recall) -- the
    final box-by-box area + SSIM check happens in stage 2.
    """
    paths = list(records.keys())
    n = len(paths)
    hash_array = np.array([records[p][0][1] for p in paths], dtype=np.uint64)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    print(f"\nGrouping {n} images by similar crop pHash (candidate threshold={threshold})...\n")

    block_size = 500
    for start in range(0, n, block_size):
        end = min(start + block_size, n)
        block = hash_array[start:end]
        xor = block[:, None] ^ hash_array[None, :]
        dist = np.unpackbits(xor.view(np.uint8), axis=-1).reshape(xor.shape[0], xor.shape[1], -1).sum(axis=-1)

        for local_i in range(end - start):
            global_i = start + local_i
            close = np.where(dist[local_i] <= threshold)[0]
            for j in close:
                if j != global_i:
                    union(global_i, int(j))

        pct = end / n * 100 if n else 100
        sys.stdout.write(f"\r  grouped {end}/{n} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line

    groups = {}
    for i in range(n):
        root = find(i)
        groups.setdefault(root, []).append(paths[i])

    return [g for g in groups.values() if len(g) > 1]


def boxes_match(boxes_a, boxes_b):
    """True if two images' box lists are the same length and every box
    pairs up (by descending area order) within AREA_TOLERANCE and passes
    SSIM at CONTENT_THRESHOLD."""
    if len(boxes_a) != len(boxes_b):
        return False
    for (area_a, _hash_a, gray_a), (area_b, _hash_b, gray_b) in zip(boxes_a, boxes_b):
        if area_a == 0 or area_b == 0:
            return False
        relative_diff = abs(area_a - area_b) / max(area_a, area_b)
        if relative_diff > AREA_TOLERANCE:
            return False
        score = ssim(gray_a, gray_b)
        if score < CONTENT_THRESHOLD:
            return False
    return True


def confirm_with_ssim(candidate_groups, records: dict):
    """
    Stage 2: within each pHash candidate group, confirm true duplicates
    using boxes_match() (area + per-box SSIM). Uses greedy
    cluster-representative matching (compare each image only against
    existing cluster representatives, not every pair) to stay fast, with
    per-image progress reporting. Every candidate group is processed here,
    regardless of size.
    """
    total_candidates = sum(len(g) for g in candidate_groups)
    print(f"\nConfirming {len(candidate_groups)} candidate group(s) "
          f"({total_candidates} images total) with SSIM (threshold={CONTENT_THRESHOLD})...\n")

    confirmed_groups = []
    checked = 0
    for group in candidate_groups:
        clusters = []  # list of lists of paths; clusters[i][0] is the representative

        for p in group:
            placed = False
            for cluster in clusters:
                if boxes_match(records[cluster[0]], records[p]):
                    cluster.append(p)
                    placed = True
                    break
            if not placed:
                clusters.append([p])

            checked += 1
            if checked % 5 == 0 or checked == total_candidates:
                pct = checked / total_candidates * 100 if total_candidates else 100
                sys.stdout.write(f"\r  confirmed {checked}/{total_candidates} candidate images ({pct:.1f}%)")
                sys.stdout.flush()

        confirmed_groups.extend(c for c in clusters if len(c) > 1)

    print()  # newline after progress line
    return confirmed_groups


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    out_dir = Path(REVIEW_DIR)
    out_images = out_dir / "images"
    out_labels = out_dir / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    records = build_box_records(images_dir, labels_dir)

    if len(records) < 2:
        print("\nNot enough labeled images to compare.")
        return

    candidate_groups = group_by_hash(records, HASH_GROUP_THRESHOLD)
    group_sizes = sorted((len(g) for g in candidate_groups), reverse=True)
    print(f"\nFound {len(candidate_groups)} candidate group(s) from pHash pre-filter.")
    if group_sizes:
        print(f"  Largest candidate group: {group_sizes[0]} images")
        print(f"  Top 5 group sizes:       {group_sizes[:5]}")

    if not candidate_groups:
        groups = []
    else:
        groups = confirm_with_ssim(candidate_groups, records)

    print(f"\nFound {len(groups)} duplicate group(s).\n")

    moved_images = 0
    for group_num, group in enumerate(groups, start=1):
        group_sorted = sorted(group, key=lambda p: p.name)
        keeper = group_sorted[0]
        losers = group_sorted[1:]

        print(f"  [group {group_num}] keeping {keeper.name}, moving {len(losers)} duplicate(s): "
              f"{', '.join(p.name for p in losers)}")

        for img_path in losers:
            dst_img = out_images / img_path.name
            shutil.move(str(img_path), str(dst_img))

            label_path = labels_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.move(str(label_path), str(out_labels / label_path.name))

            moved_images += 1

    remaining_images = sum(1 for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

    print("\nDone.")
    print(f"  Duplicate groups found:     {len(groups)}")
    print(f"  Images removed (moved out): {moved_images}")
    print(f"  Images remaining in merged: {remaining_images}")
    print(f"  Review folder:              {out_dir.resolve()}")
    print("\nNothing was deleted. Check the review folder -- if the moved images")
    print("really are unwanted duplicates, delete that folder. If any group was")
    print("wrongly matched, move those specific files back into")
    print("merged_dataset/images and .../labels.")


if __name__ == "__main__":
    main()

### `duplicates_CAHTE.py`

dedupe_by_label_area_ssim.py

In [ ]:
"""
dedupe_by_label_area_ssim.py

Compares the CROPPED LABELED REGION of each image (not the whole picture)
to find duplicates -- catches cases where the same subject was annotated
in two different images with different backgrounds/crops, lighting,
color, or compression around it.

PIPELINE:

    Load images + crop labeled boxes
       |
    CLAHE-normalize each crop (neutralizes lighting/color/compression)
       |
    Compute pHash of the NORMALIZED crop
       |
    Group images with similar crop-hashes (loose threshold, candidate filter)
       |
    Within each group: confirm with SSIM (strict, pixel-level) OR
    ORB keypoint matching + RANSAC (tolerant of shift/scale/rotation)
       |
    Write a manifest + contact sheets, THEN move confirmed duplicates

HOW A MATCH IS DECIDED (two things must be true):
  1. Same number of boxes, and each box's normalized AREA (w*h) matches
     within AREA_TOLERANCE -- cheap filter, checked first.
  2. Each box's cropped CONTENT matches via SSIM at CONTENT_THRESHOLD,
     OR via ORB+RANSAC geometric matching -- catches cases SSIM misses
     (crop shift, brightness/color shift, slight rotation) without
     losing SSIM's precision on true near-pixel matches.

SAFETY: confirmed duplicates (image + label) are MOVED to a review
folder, nothing is deleted. A manifest.json + contact sheet image are
written per group BEFORE anything is moved, so every match can be
verified visually (with its SSIM score / ORB inlier count) instead of
by eye alone.

Usage:
    pip install pillow numpy scikit-image imagehash opencv-python-headless --break-system-packages
    python dedupe_by_label_area_ssim.py
"""

import json
import shutil
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw
from skimage.metrics import structural_similarity as ssim
import imagehash

MERGED_DIR = "datasets/merged_dataset"
REVIEW_DIR = "datasets/duplicates_by_label_review_ssim"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# How close two normalized box areas (w*h, both already 0-1) need to be to
# even bother comparing pixel content. 0.02 = within 2% relative difference.
AREA_TOLERANCE = 0.02

# Stage 1 (pHash on the CLAHE-normalized crop) -- loose candidate filter.
HASH_SIZE = 8  # 8x8 = 64-bit hash
HASH_GROUP_THRESHOLD = 10  # loosened vs raw-pixel hashing to give matching margin

# Stage 2a (SSIM on the normalized crop) -- strict final confirmation
# (-1 to 1, 1.0 = structurally identical).
CONTENT_THRESHOLD = 0.99

# Stage 2b (ORB + RANSAC) -- geometric confirmation, tolerant of crop
# shift, scale, rotation, and brightness changes that SSIM is not.
ORB_N_FEATURES = 500
ORB_MIN_GOOD_MATCHES = 8      # min Lowe-ratio-passing matches before trying homography
ORB_MIN_INLIERS = 10          # min RANSAC inliers to call it a match
ORB_MIN_INLIER_RATIO = 0.6    # inliers / good matches

# Crops are resized to this fixed size for pHash/SSIM/ORB, since box crops
# naturally come out at different pixel sizes.
COMPARE_SIZE = (128, 128)

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
_orb = cv2.ORB_create(nfeatures=ORB_N_FEATURES)
_bf = cv2.BFMatcher(cv2.NORM_HAMMING)


def parse_boxes(label_path: Path):
    """Return list of (class_id, x_center, y_center, w, h), all normalized
    0-1, from a YOLO .txt label file."""
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        class_id, xc, yc, w, h = parts[:5]
        boxes.append((int(class_id), float(xc), float(yc), float(w), float(h)))
    return boxes


def crop_box(im: Image.Image, xc, yc, w, h):
    """Crop a normalized YOLO box out of a PIL image. Computes pixel
    width/height first, then derives left/top from the center and that
    fixed size (rather than rounding all four edges independently) --
    otherwise two boxes with identical normalized w/h can end up 1px
    different in size after rounding, throwing off the comparison."""
    img_w, img_h = im.size
    box_w = round(w * img_w)
    box_h = round(h * img_h)
    center_x = round(xc * img_w)
    center_y = round(yc * img_h)

    left = max(0, center_x - box_w // 2)
    top = max(0, center_y - box_h // 2)
    right = min(img_w, left + box_w)
    bottom = min(img_h, top + box_h)

    if right <= left or bottom <= top:
        return None
    return im.crop((left, top, right, bottom))


def normalize_gray(crop: Image.Image) -> np.ndarray:
    """Grayscale + resize + CLAHE contrast normalization. This neutralizes
    lighting/color/compression differences BEFORE hashing, SSIM, or ORB
    ever see the crop, so all three stages compare on a level footing."""
    gray = np.array(crop.convert("L").resize(COMPARE_SIZE, Image.LANCZOS), dtype=np.uint8)
    return _clahe.apply(gray)


def build_box_records(images_dir: Path, labels_dir: Path):
    """
    For every image that has a label file, crop out each box, CLAHE-
    normalize it, and compute its pHash (stage 1), normalized grayscale
    array (stage 2a/SSIM), and ORB keypoints+descriptors (stage 2b).
    Returns {img_path: [(area, hash_int, norm_gray, keypoints, descriptors), ...]}
    sorted by area descending per image.
    """
    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(image_paths)
    print(f"Cropping + normalizing labeled regions for {total} images...\n")

    records = {}
    for i, img_path in enumerate(image_paths, start=1):
        label_path = labels_dir / f"{img_path.stem}.txt"
        boxes = parse_boxes(label_path)
        if not boxes:
            continue

        try:
            with Image.open(img_path) as im:
                im = im.convert("RGB")
                box_records = []
                for (_cls, xc, yc, w, h) in boxes:
                    crop = crop_box(im, xc, yc, w, h)
                    if crop is None:
                        continue
                    area = w * h
                    norm_gray = normalize_gray(crop)
                    h_hash = int(str(imagehash.phash(Image.fromarray(norm_gray), hash_size=HASH_SIZE)), 16)
                    keypoints, descriptors = _orb.detectAndCompute(norm_gray, None)
                    box_records.append((area, h_hash, norm_gray, keypoints, descriptors))
                box_records.sort(key=lambda r: r[0], reverse=True)
                if box_records:
                    records[img_path] = box_records
        except Exception as e:
            print(f"\n[WARNING] Could not process {img_path.name}: {e}")

        pct = i / total * 100 if total else 100
        sys.stdout.write(f"\r  processed {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line
    return records


def group_by_hash(records: dict, threshold: int):
    """
    Stage 1: cheap candidate filter, using the LARGEST box's pHash per
    image (the main subject) as a representative signature. Vectorized
    with numpy in row blocks. Deliberately loose (high recall) -- the
    final area + SSIM/ORB check happens in stage 2.
    """
    paths = list(records.keys())
    n = len(paths)
    hash_array = np.array([records[p][0][1] for p in paths], dtype=np.uint64)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    print(f"\nGrouping {n} images by similar crop pHash (candidate threshold={threshold})...\n")

    block_size = 500
    for start in range(0, n, block_size):
        end = min(start + block_size, n)
        block = hash_array[start:end]
        xor = block[:, None] ^ hash_array[None, :]
        dist = np.unpackbits(xor.view(np.uint8), axis=-1).reshape(xor.shape[0], xor.shape[1], -1).sum(axis=-1)

        for local_i in range(end - start):
            global_i = start + local_i
            close = np.where(dist[local_i] <= threshold)[0]
            for j in close:
                if j != global_i:
                    union(global_i, int(j))

        pct = end / n * 100 if n else 100
        sys.stdout.write(f"\r  grouped {end}/{n} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line

    groups = {}
    for i in range(n):
        root = find(i)
        groups.setdefault(root, []).append(paths[i])

    return [g for g in groups.values() if len(g) > 1]


def orb_match(kp1, desc1, kp2, desc2):
    """Match ORB descriptors with a Lowe ratio test, then verify the match
    geometrically with RANSAC homography. Tolerant of crop shift, scale,
    rotation, and brightness differences that raw SSIM is not.
    Returns (matched: bool, inlier_count: int, good_match_count: int)."""
    if desc1 is None or desc2 is None or len(desc1) < 2 or len(desc2) < 2:
        return False, 0, 0

    raw_matches = _bf.knnMatch(desc1, desc2, k=2)
    good = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

    if len(good) < ORB_MIN_GOOD_MATCHES:
        return False, 0, len(good)

    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    _, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    if mask is None:
        return False, 0, len(good)

    inliers = int(mask.sum())
    inlier_ratio = inliers / len(good)
    matched = inliers >= ORB_MIN_INLIERS and inlier_ratio >= ORB_MIN_INLIER_RATIO
    return matched, inliers, len(good)


def compare_boxes(box_a, box_b):
    """Compare one box pair. Returns a dict with the match decision plus
    evidence (SSIM score / ORB inlier count) -- used both to decide the
    match and to annotate the contact sheet / manifest for visual review."""
    area_a, _hash_a, gray_a, kp_a, desc_a = box_a
    area_b, _hash_b, gray_b, kp_b, desc_b = box_b

    result = {"matched": False, "method": None, "ssim": None,
              "orb_inliers": None, "orb_good_matches": None}

    if area_a == 0 or area_b == 0:
        return result
    if abs(area_a - area_b) / max(area_a, area_b) > AREA_TOLERANCE:
        return result

    score = float(ssim(gray_a, gray_b))
    result["ssim"] = round(score, 4)
    if score >= CONTENT_THRESHOLD:
        result["matched"] = True
        result["method"] = "ssim"
        return result

    orb_matched, inliers, good = orb_match(kp_a, desc_a, kp_b, desc_b)
    result["orb_inliers"] = inliers
    result["orb_good_matches"] = good
    if orb_matched:
        result["matched"] = True
        result["method"] = "orb"
    return result


def boxes_match(boxes_a, boxes_b):
    """True (+ per-box evidence) only if EVERY box pairs up within
    AREA_TOLERANCE and is confirmed by SSIM or ORB+RANSAC."""
    if len(boxes_a) != len(boxes_b):
        return False, []
    evidence = []
    for box_a, box_b in zip(boxes_a, boxes_b):
        result = compare_boxes(box_a, box_b)
        if not result["matched"]:
            return False, []
        evidence.append(result)
    return True, evidence


def confirm_matches(candidate_groups, records: dict):
    """
    Stage 2: within each pHash candidate group, confirm true duplicates
    using boxes_match() (area + per-box SSIM/ORB). Uses greedy
    cluster-representative matching (compare each image only against
    existing cluster representatives, not every pair) to stay fast.
    Every candidate group is processed, regardless of size.
    Returns a list of clusters: {"paths": [...], "evidence": {path: [...]}}
    """
    total_candidates = sum(len(g) for g in candidate_groups)
    print(f"\nConfirming {len(candidate_groups)} candidate group(s) "
          f"({total_candidates} images total) with SSIM + ORB...\n")

    confirmed = []
    checked = 0
    for group in candidate_groups:
        clusters = []  # each: {"paths": [rep, ...], "evidence": {path: evidence_list}}

        for p in group:
            placed = False
            for cluster in clusters:
                rep = cluster["paths"][0]
                matched, evidence = boxes_match(records[rep], records[p])
                if matched:
                    cluster["paths"].append(p)
                    cluster["evidence"][p] = evidence
                    placed = True
                    break
            if not placed:
                clusters.append({"paths": [p], "evidence": {}})

            checked += 1
            if checked % 5 == 0 or checked == total_candidates:
                pct = checked / total_candidates * 100 if total_candidates else 100
                sys.stdout.write(f"\r  confirmed {checked}/{total_candidates} candidate images ({pct:.1f}%)")
                sys.stdout.flush()

        confirmed.extend(c for c in clusters if len(c["paths"]) > 1)

    print()  # newline after progress line
    return confirmed


def build_contact_sheet(cluster, images_dir: Path, out_path: Path):
    """Render a duplicate group as a row of side-by-side thumbnails, each
    labeled with filename + match evidence, so a group can be verified
    visually instead of by eye alone."""
    paths = cluster["paths"]
    thumb_size = 200
    padding = 10
    label_h = 80
    width = len(paths) * (thumb_size + padding) + padding
    height = thumb_size + label_h + padding * 2
    sheet = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(sheet)

    for i, p in enumerate(paths):
        x = padding + i * (thumb_size + padding)
        try:
            with Image.open(images_dir / p.name) as im:
                im = im.convert("RGB")
                im.thumbnail((thumb_size, thumb_size))
                sheet.paste(im, (x, padding))
        except Exception:
            pass

        if i == 0:
            label = f"{p.name}\n(kept)"
        else:
            ev = cluster["evidence"].get(p, [])
            parts = []
            for box_i, e in enumerate(ev):
                if e["method"] == "ssim":
                    parts.append(f"box{box_i}: SSIM {e['ssim']}")
                elif e["method"] == "orb":
                    parts.append(f"box{box_i}: ORB {e['orb_inliers']}in/{e['orb_good_matches']}m")
            label = f"{p.name}\n" + "\n".join(parts)

        draw.multiline_text((x, thumb_size + padding + 5), label, fill="black")

    sheet.save(out_path)


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    out_dir = Path(REVIEW_DIR)
    out_images = out_dir / "images"
    out_labels = out_dir / "labels"
    contact_sheets_dir = out_dir / "contact_sheets"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    contact_sheets_dir.mkdir(parents=True, exist_ok=True)

    records = build_box_records(images_dir, labels_dir)

    if len(records) < 2:
        print("\nNot enough labeled images to compare.")
        return

    candidate_groups = group_by_hash(records, HASH_GROUP_THRESHOLD)
    group_sizes = sorted((len(g) for g in candidate_groups), reverse=True)
    print(f"\nFound {len(candidate_groups)} candidate group(s) from pHash pre-filter.")
    if group_sizes:
        print(f"  Largest candidate group: {group_sizes[0]} images")
        print(f"  Top 5 group sizes:       {group_sizes[:5]}")

    clusters = confirm_matches(candidate_groups, records) if candidate_groups else []

    print(f"\nFound {len(clusters)} duplicate group(s).\n")

    manifest = []
    moved_images = 0
    for group_num, cluster in enumerate(clusters, start=1):
        cluster["paths"].sort(key=lambda p: p.name)
        keeper = cluster["paths"][0]
        losers = cluster["paths"][1:]

        print(f"  [group {group_num}] keeping {keeper.name}, moving {len(losers)} duplicate(s): "
              f"{', '.join(p.name for p in losers)}")

        sheet_path = contact_sheets_dir / f"group_{group_num:04d}.jpg"
        build_contact_sheet(cluster, images_dir, sheet_path)

        manifest.append({
            "group": group_num,
            "kept": keeper.name,
            "moved": [
                {"file": p.name, "evidence": cluster["evidence"].get(p, [])}
                for p in losers
            ],
            "contact_sheet": str(sheet_path.relative_to(out_dir)),
        })

        for img_path in losers:
            dst_img = out_images / img_path.name
            shutil.move(str(img_path), str(dst_img))

            label_path = labels_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.move(str(label_path), str(out_labels / label_path.name))

            moved_images += 1

    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

    remaining_images = sum(1 for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

    print("\nDone.")
    print(f"  Duplicate groups found:     {len(clusters)}")
    print(f"  Images removed (moved out): {moved_images}")
    print(f"  Images remaining in merged: {remaining_images}")
    print(f"  Review folder:              {out_dir.resolve()}")
    print(f"  Contact sheets:             {contact_sheets_dir.resolve()}")
    print(f"  Manifest (match evidence):  {(out_dir / 'manifest.json').resolve()}")
    print("\nNothing was deleted. Open the contact sheets in the review folder to")
    print("verify matches visually (SSIM score / ORB inlier count shown per box)")
    print("before deleting the review folder for good. If any group was wrongly")
    print("matched, move those specific files back into merged_dataset/images")
    print("and .../labels.")


if __name__ == "__main__":
    main()

### `empty_labels.py`

move_empty_labels.py

In [ ]:
"""
move_empty_labels.py

Finds every image whose label file is EMPTY (0 bytes, or a .txt with no
actual annotation lines) and moves that image + its label out of the
merged dataset into a separate folder. Nothing is deleted.

Usage:
    python move_empty_labels.py
"""

import shutil
import sys
from pathlib import Path

MERGED_DIR = "datasets/merged_dataset"
EMPTY_LABELS_DIR = "datasets/empty_labels_review"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def is_empty_label(label_path: Path) -> bool:
    if label_path.stat().st_size == 0:
        return True
    if label_path.suffix == ".txt":
        return not label_path.read_text().strip()
    return False


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    out_dir = Path(EMPTY_LABELS_DIR)
    out_images = out_dir / "images"
    out_labels = out_dir / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(image_paths)
    print(f"Checking {total} images for empty labels...\n")

    moved = 0
    for i, img_path in enumerate(image_paths, start=1):
        label_path = labels_dir / f"{img_path.stem}.txt"

        if label_path.exists() and is_empty_label(label_path):
            shutil.move(str(img_path), str(out_images / img_path.name))
            shutil.move(str(label_path), str(out_labels / label_path.name))
            moved += 1

        pct = i / total * 100 if total else 100
        sys.stdout.write(f"\r  checked {i}/{total} ({pct:.1f}%) | moved so far: {moved}")
        sys.stdout.flush()

    print()  # newline after progress line

    remaining_images = sum(1 for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

    print("\nDone.")
    print(f"  Images with empty labels moved: {moved}")
    print(f"  Images remaining in merged:      {remaining_images}")
    print(f"  Review folder:                   {out_dir.resolve()}")


if __name__ == "__main__":
    main()

### `extract_unknown_class.py`

extract_unknown_class.py

In [ ]:
"""
extract_unknown_class.py

Scans the MERGED dataset (produced by merge_datasets_yaml.py) for every
image that has at least one box labeled "unknown_0" (class ID 1 in
UNIFIED_CLASSES), and copies those images + their label files into a
separate review folder so you can eyeball them and decide whether they're
actually rats, something else, or junk boxes to delete.

It also crops out just the "unknown_0" boxes themselves as small image
snippets (in a crops/ subfolder), since that's usually faster to review
than opening full images and hunting for the box.

Usage:
    pip install pillow --break-system-packages   (if not already installed)
    python extract_unknown_class.py
"""

import shutil
import sys
from pathlib import Path
from PIL import Image

MERGED_DIR = "datasets/merged_dataset"      # where merge_datasets_yaml.py wrote its output
REVIEW_DIR = "datasets/unknown_0_review"    # where flagged images get copied

UNKNOWN_CLASS_ID = 1   # "unknown_0" is index 1 in UNIFIED_CLASSES = ["rat", "unknown_0"]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def yolo_box_to_pixels(x_center, y_center, w, h, img_w, img_h):
    """Convert normalized YOLO box (0-1 range) to pixel (left, top, right, bottom)."""
    x_center *= img_w
    y_center *= img_h
    w *= img_w
    h *= img_h
    left = max(0, int(x_center - w / 2))
    top = max(0, int(y_center - h / 2))
    right = min(img_w, int(x_center + w / 2))
    bottom = min(img_h, int(y_center + h / 2))
    return left, top, right, bottom


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    out_dir = Path(REVIEW_DIR)
    out_images = out_dir / "images"
    out_labels = out_dir / "labels"
    out_crops = out_dir / "crops"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    out_crops.mkdir(parents=True, exist_ok=True)

    flagged_images = 0
    flagged_boxes = 0

    # Pre-pass just to get a total count for the progress line below.
    all_label_files = sorted(p for p in labels_dir.iterdir() if p.suffix.lower() == ".txt")
    total_labels = len(all_label_files)
    print(f"Scanning {total_labels} label files for unknown_0 boxes...\n")

    for scanned, label_path in enumerate(all_label_files, start=1):
        lines = label_path.read_text().splitlines()
        unknown_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            if int(parts[0]) == UNKNOWN_CLASS_ID:
                unknown_lines.append(parts)

        # Live progress line, overwritten in place (no scroll spam).
        pct = (scanned / total_labels * 100) if total_labels else 100
        sys.stdout.write(
            f"\r  scanned {scanned}/{total_labels} ({pct:.1f}%) "
            f"| flagged so far: {flagged_images}"
        )
        sys.stdout.flush()

        if not unknown_lines:
            continue  # this image has no unknown_0 boxes, skip it

        # Find the matching image (same stem, any image extension)
        stem = label_path.stem
        matches = [p for p in images_dir.glob(f"{stem}.*") if p.suffix.lower() in IMAGE_EXTS]
        if not matches:
            print(f"[WARNING] Label {label_path.name} has unknown_0 boxes but no matching image found, skipping.")
            continue
        img_path = matches[0]

        # Copy the full image + its full label file (with ALL boxes, not just
        # unknown_0) into the review folder, so you have full context.
        shutil.copy2(img_path, out_images / img_path.name)
        shutil.copy2(label_path, out_labels / label_path.name)
        flagged_images += 1

        # Also save a cropped snippet of just each unknown_0 box, for fast review.
        try:
            with Image.open(img_path) as im:
                img_w, img_h = im.size
                for i, parts in enumerate(unknown_lines):
                    _, xc, yc, w, h = parts[:5]
                    box = yolo_box_to_pixels(float(xc), float(yc), float(w), float(h), img_w, img_h)
                    crop = im.crop(box)
                    crop_name = f"{stem}_box{i}{img_path.suffix}"
                    crop.save(out_crops / crop_name)
                    flagged_boxes += 1
        except Exception as e:
            print(f"[WARNING] Could not crop boxes from {img_path.name}: {e}")

    print()  # newline after the in-place progress line
    print("Done.")
    print(f"  Images with unknown_0 boxes: {flagged_images}")
    print(f"  Total unknown_0 boxes:       {flagged_boxes}")
    print(f"  Full images + labels copied to: {out_images.resolve()} / {out_labels.resolve()}")
    print(f"  Cropped box snippets saved to:  {out_crops.resolve()}")
    print("\nReview tip: browse crops/ first (fastest way to see just the boxed")
    print("region), then open the matching full image in images/ if you need")
    print("more context on any particular one.")


if __name__ == "__main__":
    main()

### `fixing_labels.py`

If it's already a normal bounding box (5 values), leave it alone

In [ ]:
import os
from pathlib import Path

LABELS_DIR = "datasets/merged_dataset/labels"

def convert_polygons_to_bboxes(labels_path):
    for txt_file in Path(labels_path).glob("*.txt"):
        with open(txt_file, 'r') as f:
            lines = f.readlines()
        
        new_lines = []
        changed = False
        
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
                
            # If it's already a normal bounding box (5 values), leave it alone
            if len(parts) == 5:
                new_lines.append(line.strip())
                continue
            
            # If it's a polygon/OBB (class + pairs of coordinates)
            try:
                cls_id = parts[0]
                coords = [float(p) for p in parts[1:]]
                
                # Separate X and Y coordinates
                xs = coords[0::2] # Even indices are X
                ys = coords[1::2] # Odd indices are Y
                
                # Calculate bounding box
                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                
                # Convert to YOLO format (x_center, y_center, width, height)
                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2
                width = x_max - x_min
                height = y_max - y_min
                
                new_line = f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                new_lines.append(new_line)
                changed = True
                
            except Exception as e:
                print(f"Error processing {txt_file.name}: {e}")
                new_lines.append(line.strip()) # Keep original if it fails

        # Overwrite the file if we converted any polygons
        if changed:
            with open(txt_file, 'w') as f:
                f.write("\n".join(new_lines) + "\n")
            print(f"Converted polygons to bboxes in: {txt_file.name}")

if __name__ == "__main__":
    convert_polygons_to_bboxes(LABELS_DIR)
    print("Done!")

### `generate_backgrounds.py`

generate_backgrounds.py

In [ ]:
"""
generate_backgrounds.py

Synthesize BACKGROUND (no-rat) images for Edge Impulse from your EXISTING
labeled dataset, for when you have no separate empty-scene footage.

Idea: every image contains a rat, but the rat usually covers only part of the
frame. The YOLO boxes tell us exactly where the rat is NOT, so we crop a large
rat-free square -> a real "empty station" background (same floor, food bowl,
shadows, lighting), with zero rat. We then apply the SAME ESP32-CAM
domain-randomization look as the positives (so style can't be used as a shortcut)
and write them into datasets/ei_curated/{training,testing}/ in the Edge Impulse
CLI label format (empty boundingBoxes = background sample).

Run:   python generate_backgrounds.py
Then:  edge-impulse-uploader --directory datasets/ei_curated
       (only the new bg_*.jpg upload; existing images are skipped by hash)
       -> Studio: regenerate features + retrain.

Requires: pillow, numpy, imagehash
"""

import json
import random
from pathlib import Path

import numpy as np
from PIL import Image
import imagehash

# ----------------------------- configuration ------------------------------
IMAGES_DIR = Path("datasets/merged_dataset/images")
LABELS_DIR = Path("datasets/merged_dataset/labels")
CURATED_DIR = Path("datasets/ei_curated")      # where the positives already live
PREFIX = "bg_"

TARGET_BG = 600            # total backgrounds to create (~12-15% of positives)
TRAIN_RATIO = 0.8          # split, same as the positives
MIN_SIDE = 120             # min side (px) of a usable rat-free square
MARGIN = 6                 # px kept away from any rat box
PHASH_THRESHOLD = 8        # dedup: skip backgrounds too similar to accepted ones
RANDOM_SEED = 42
JPEG_QUALITY = 90

# match the positives' style distribution / ESP32-CAM look
P_CLEAN = 1.0 / 3.0        # keep as clean color (like the untouched originals)
P_GRAYSCALE = 0.7
P_DARKEN = 0.6
P_LOWRES = 0.7
P_NOISE = 0.5
P_HFLIP = 0.5
# --------------------------------------------------------------------------


def read_yolo_boxes(txt_path: Path):
    """Return a list of (xc, yc, w, h) normalized floats."""
    boxes = []
    if not txt_path.exists():
        return boxes
    for line in txt_path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        try:
            _cls = int(float(parts[0]))
            xc, yc, w, h = (float(v) for v in parts[1:])
        except ValueError:
            continue
        boxes.append((xc, yc, w, h))
    return boxes


def rat_free_square(boxes, W, H):
    """Largest rat-free square as (left, top, side), or None.

    Any rat's left edge is >= union-left, so the strip left of the union box is
    guaranteed rat-free; same reasoning for right/top/bottom strips.
    """
    if not boxes:
        side = min(W, H)
        return ((W - side) // 2, (H - side) // 2, side)

    ux0, uy0, ux1, uy1 = float(W), float(H), 0.0, 0.0
    for xc, yc, w, h in boxes:
        ux0 = min(ux0, (xc - w / 2) * W - MARGIN)
        uy0 = min(uy0, (yc - h / 2) * H - MARGIN)
        ux1 = max(ux1, (xc + w / 2) * W + MARGIN)
        uy1 = max(uy1, (yc + h / 2) * H + MARGIN)
    ux0 = max(0.0, ux0); uy0 = max(0.0, uy0)
    ux1 = min(float(W), ux1); uy1 = min(float(H), uy1)

    candidates = [
        (0, 0, max(0.0, ux0), H),   # left strip
        (min(float(W), ux1), 0, W, H),  # right strip
        (0, 0, W, max(0.0, uy0)),   # top strip
        (0, min(float(H), uy1), W, H),  # bottom strip
    ]
    best = None
    for x0, y0, x1, y1 in candidates:
        rw, rh = x1 - x0, y1 - y0
        side = int(min(rw, rh))
        if side >= MIN_SIDE and (best is None or side > best[2]):
            cx0 = int(x0 + (rw - side) / 2)
            cy0 = int(y0 + (rh - side) / 2)
            best = (cx0, cy0, side)
    return best


def augment(im: Image.Image) -> Image.Image:
    """Apply the ESP32-CAM domain-randomization look to a background crop."""
    if random.random() < P_HFLIP:
        im = im.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < P_GRAYSCALE:
        im = im.convert("L").convert("RGB")
    arr = np.asarray(im).astype(np.float32)
    if random.random() < P_DARKEN:
        gamma = random.uniform(1.3, 2.2)
        arr = 255.0 * (arr / 255.0) ** gamma
    if random.random() < P_NOISE:
        arr = arr + np.random.normal(0, random.uniform(4, 14), arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    out = Image.fromarray(arr)
    if random.random() < P_LOWRES:
        w, h = out.size
        f = random.uniform(0.35, 0.6)
        out = out.resize((max(1, int(w * f)), max(1, int(h * f))), Image.BILINEAR)
        out = out.resize((w, h), Image.NEAREST)
    return out


def next_index(*dirs) -> int:
    mx = 0
    for d in dirs:
        for p in d.glob(f"{PREFIX}*.jpg"):
            s = p.stem[len(PREFIX):]
            if s.isdigit():
                mx = max(mx, int(s))
    return mx + 1


def load_labels(path: Path):
    if path.exists():
        data = json.loads(path.read_text())
        return data, data.get("boundingBoxes", {})
    return {"version": 1, "type": "bounding-box-labels", "boundingBoxes": {}}, {}


def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    train_dir = CURATED_DIR / "training"
    test_dir = CURATED_DIR / "testing"
    for d in (train_dir, test_dir):
        d.mkdir(parents=True, exist_ok=True)

    train_data, train_bb = load_labels(train_dir / "bounding_boxes.labels")
    test_data, test_bb = load_labels(test_dir / "bounding_boxes.labels")
    if not train_bb:
        print("WARNING: no positives found in datasets/ei_curated/training - "
              "run curate_and_augment.py first.")

    imgs = sorted(IMAGES_DIR.glob("*.jpg"))
    random.shuffle(imgs)
    print(f"Scanning up to {len(imgs)} images for rat-free regions "
          f"(target {TARGET_BG})...")

    accepted_hashes = []
    crops = []
    for p in imgs:
        if len(crops) >= TARGET_BG:
            break
        boxes = read_yolo_boxes(LABELS_DIR / f"{p.stem}.txt")
        try:
            im = Image.open(p).convert("RGB")
        except Exception:
            continue
        W, H = im.size
        sq = rat_free_square(boxes, W, H)
        if sq is None:
            continue
        x0, y0, side = sq
        crop = im.crop((x0, y0, x0 + side, y0 + side))
        hsh = imagehash.phash(crop)
        if any((hsh - h) <= PHASH_THRESHOLD for h in accepted_hashes):
            continue
        accepted_hashes.append(hsh)
        crops.append(crop)
        if len(crops) % 100 == 0:
            print(f"  collected {len(crops)}/{TARGET_BG}")

    print(f"Collected {len(crops)} diverse rat-free backgrounds.")
    if not crops:
        print("No usable rat-free regions found. Lower MIN_SIDE and retry.")
        return

    random.shuffle(crops)
    n_train = int(len(crops) * TRAIN_RATIO)
    idx = next_index(train_dir, test_dir)
    added = {"training": 0, "testing": 0}

    for i, crop in enumerate(crops):
        if i < n_train:
            out_dir, bb, split = train_dir, train_bb, "training"
        else:
            out_dir, bb, split = test_dir, test_bb, "testing"
        out = crop if random.random() < P_CLEAN else augment(crop)
        name = f"{PREFIX}{idx:06d}.jpg"
        idx += 1
        out.save(out_dir / name, "JPEG", quality=JPEG_QUALITY)
        bb[name] = []                       # empty boxes = background sample
        added[split] += 1

    for data, bb in ((train_data, train_bb), (test_data, test_bb)):
        data["version"] = 1
        data["type"] = "bounding-box-labels"
        data["boundingBoxes"] = bb
    (train_dir / "bounding_boxes.labels").write_text(json.dumps(train_data))
    (test_dir / "bounding_boxes.labels").write_text(json.dumps(test_data))

    positives = sum(1 for k in list(train_bb) + list(test_bb)
                    if not k.startswith(PREFIX))
    backgrounds = sum(1 for k in list(train_bb) + list(test_bb)
                      if k.startswith(PREFIX))
    ratio = backgrounds / positives * 100 if positives else 0.0

    print("\nDone.")
    print(f"  Backgrounds added : training {added['training']}, "
          f"testing {added['testing']}")
    print(f"  Positives         : {positives}")
    print(f"  Backgrounds total : {backgrounds} ({ratio:.1f}% of positives)")
    if ratio < 10:
        print("  TIP: raise TARGET_BG for more (~10-20% recommended).")
    elif ratio > 30:
        print("  NOTE: >30% backgrounds can start to hurt recall.")
    print("\nNext steps:")
    print("  1) edge-impulse-uploader --directory datasets/ei_curated")
    print("     (only new bg_*.jpg upload; existing images skipped by hash)")
    print("  2) Studio -> Impulse -> Image -> Generate features")
    print("  3) Retrain, then Model testing to compare precision.")


if __name__ == "__main__":
    main()


### `merge_datasets.py`

merge_datasets_yaml.py

In [ ]:
"""
merge_datasets_yaml.py

Merges multiple YOLO-format datasets (images/ + labels/) into one combined
dataset, using each dataset's data.yaml to figure out what its class IDs
actually mean, then rewriting every label file so class IDs are consistent
across the merged set.

Why this exists: each of your datasets was exported separately, so class ID 0
does NOT mean the same thing in every one. This script fixes that instead of
just dumping label files together blindly.

Usage:
    1. Edit DATASETS with the path to each dataset folder.
    2. Edit OUTPUT_DIR to where you want the merged result written.
    3. Double check ALIASES / MANUAL_CLASSES / UNIFIED_CLASSES match your data.
    4. pip install pyyaml (if not already installed)
    5. Run: python merge_datasets_yaml.py
"""

import shutil
import sys
import yaml
from pathlib import Path

# ---------------------------------------------------------------------
# 1. DATASETS: tag -> folder path
# ---------------------------------------------------------------------
DATASETS = {
    "merged": "datasets/merged",
    "kaggle": "datasets/dataset_kaggle_1",
    "ratdet": "datasets/rat_detection.v4i.yolov5pytorch",
}

OUTPUT_DIR = "datasets/merged_dataset"

SPLIT_SUBFOLDERS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# ---------------------------------------------------------------------
# 2. MANUAL_CLASSES: for any dataset with NO data.yaml, hardcode its
#    class list here. Position in the list = the class ID that dataset's
#    .txt label files actually use. (Empty now -- none of the current
#    three datasets need this; add an entry here if that changes.)
# ---------------------------------------------------------------------
MANUAL_CLASSES = {}

# ---------------------------------------------------------------------
# 3. ALIASES: raw class name (exactly as it appears in that dataset's
#    data.yaml "names:" list) -> canonical name to merge it into.
#
#    Any raw name not listed here is assumed to already be canonical
#    (e.g. "rat" maps to itself).
# ---------------------------------------------------------------------
ALIASES = {
    "mouse": "rat",   # kaggle dataset's class -- same as rat for this project
}

# Final class list for the merged dataset. Position here = the class ID
# that will be written into every remapped label file. unknown_0 has been
# removed -- it was already manually reviewed and folded into "rat" in an
# earlier pass, so it's no longer needed here.
UNIFIED_CLASSES = ["rat"]


def load_classes(tag: str, dataset_path: Path):
    """
    Return this dataset's ordered raw class name list (position = its
    original class ID), from data.yaml (or MANUAL_CLASSES if the dataset
    has no yaml at all).
    """
    if tag in MANUAL_CLASSES:
        return MANUAL_CLASSES[tag]

    yaml_path = dataset_path / "data.yaml"
    if not yaml_path.exists():
        print(f"[WARNING] No data.yaml for '{tag}' and no MANUAL_CLASSES entry "
              f"-- its labels will NOT be remapped (all lines dropped).")
        return []

    with open(yaml_path, "r", encoding="utf-8-sig") as f:
        data = yaml.safe_load(f)
    if data is None:
        print(f"[WARNING] '{tag}'s data.yaml is empty or malformed (loaded as None) -- "
              f"check {yaml_path} by hand. Its labels will NOT be remapped (all lines dropped).")
        return []
    names = data.get("names", [])
    if not names:
        print(f"[WARNING] '{tag}'s data.yaml has no 'names:' list -- "
              f"its labels will NOT be remapped (all lines dropped).")
    return names


def build_remap(raw_names):
    """
    Build {old_class_id: new_class_id} for one dataset from its raw class
    name list. Any class that can't be resolved to something in
    UNIFIED_CLASSES is dropped (its boxes will be skipped, with a warning).
    """
    remap = {}
    for old_id, raw_name in enumerate(raw_names):
        canonical = ALIASES.get(raw_name, raw_name)
        if canonical not in UNIFIED_CLASSES:
            print(f"    [WARNING] class '{raw_name}' (id {old_id}) -> "
                  f"'{canonical}' is not in UNIFIED_CLASSES, its boxes will "
                  f"be dropped. Add it to ALIASES/UNIFIED_CLASSES if that's wrong.")
            continue
        remap[old_id] = UNIFIED_CLASSES.index(canonical)
    return remap


def remap_label_file(src_path: Path, dst_path: Path, remap: dict):
    """
    Copy one label file, rewriting YOLO .txt class IDs per `remap`.
    Lines whose class ID has no entry in remap (dropped class) are skipped.
    Non-.txt labels (xml/json/etc.) are copied unchanged -- ID remapping
    only applies to YOLO .txt format.
    """
    if src_path.suffix.lower() != ".txt":
        shutil.copy2(src_path, dst_path)
        return

    out_lines = []
    for line in src_path.read_text().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        old_id = int(parts[0])
        if old_id not in remap:
            continue
        parts[0] = str(remap[old_id])
        out_lines.append(" ".join(parts))

    dst_path.write_text("\n".join(out_lines) + ("\n" if out_lines else ""))


def count_images(dataset_path: Path):
    """Quick pre-pass: count how many images this dataset has, across all
    its split subfolders, so we can show an overall progress percentage."""
    total = 0
    for img_dir, _ in find_split_dirs(dataset_path):
        total += sum(1 for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    return total


def find_split_dirs(dataset_path: Path):
    """
    Prefer a FLAT structure (dataset_path/images, dataset_path/labels) if
    it exists -- that's what your current datasets actually look like.
    Only fall back to checking train/valid/test split subfolders if the
    flat structure isn't there.
    """
    flat_img_dir = dataset_path / "images"
    if flat_img_dir.exists():
        yield flat_img_dir, dataset_path / "labels"
        return

    for split in SPLIT_SUBFOLDERS:
        base = dataset_path / split
        img_dir = base / "images"
        lbl_dir = base / "labels"
        if img_dir.exists():
            yield img_dir, lbl_dir


def merge():
    out_images = Path(OUTPUT_DIR) / "images"
    out_labels = Path(OUTPUT_DIR) / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    total_images = 0
    total_labels = 0
    missing_labels = []

    # Pre-pass: count everything upfront so we can show a running
    # "X / grand_total" and percentage instead of the terminal looking frozen.
    print("Scanning datasets to count files...")
    valid_datasets = [(tag, Path(p)) for tag, p in DATASETS.items() if Path(p).exists()]
    for tag, p in DATASETS.items():
        if not Path(p).exists():
            print(f"[WARNING] Dataset path not found, skipping: {p}")
    per_dataset_counts = {tag: count_images(path) for tag, path in valid_datasets}
    grand_total = sum(per_dataset_counts.values())
    print(f"Found {grand_total} images across {len(valid_datasets)} datasets.\n")

    processed = 0
    for tag, dataset_path in valid_datasets:
        raw_names = load_classes(tag, dataset_path)
        remap = build_remap(raw_names)
        print(f"[{tag}] classes: {raw_names} -> remap {remap}")
        dataset_total = per_dataset_counts[tag]

        dataset_count = 0
        for img_dir, lbl_dir in find_split_dirs(dataset_path):
            for img_path in sorted(img_dir.iterdir()):
                if img_path.suffix.lower() not in IMAGE_EXTS:
                    continue

                new_stem = f"{tag}_{img_path.stem}"
                new_img_path = out_images / f"{new_stem}{img_path.suffix}"
                shutil.copy2(img_path, new_img_path)

                label_path = None
                if lbl_dir.exists():
                    matches = list(lbl_dir.glob(f"{img_path.stem}.*"))
                    if matches:
                        label_path = matches[0]

                if label_path:
                    new_lbl_path = out_labels / f"{new_stem}{label_path.suffix}"
                    remap_label_file(label_path, new_lbl_path, remap)
                    total_labels += 1
                else:
                    missing_labels.append(str(img_path))

                dataset_count += 1
                total_images += 1
                processed += 1

                # Live progress line, overwritten in place (no scroll spam).
                # Prints every image; on very large datasets you can change
                # the modulo below (e.g. `if processed % 25 == 0`) to update
                # less often.
                pct = (processed / grand_total * 100) if grand_total else 100
                sys.stdout.write(
                    f"\r  [{tag}] {dataset_count}/{dataset_total} in this dataset "
                    f"| overall {processed}/{grand_total} ({pct:.1f}%)"
                )
                sys.stdout.flush()

        print()  # newline after this dataset's progress line
        print(f"[{tag}] copied {dataset_count} images\n")

    # Write out the unified data.yaml for the merged dataset
    merged_yaml = {"nc": len(UNIFIED_CLASSES), "names": UNIFIED_CLASSES}
    with open(Path(OUTPUT_DIR) / "data.yaml", "w") as f:
        yaml.safe_dump(merged_yaml, f)

    print("Merge complete.")
    print(f"  Total images copied: {total_images}")
    print(f"  Total labels copied: {total_labels}")
    print(f"  Unified classes:     {UNIFIED_CLASSES}")
    print(f"  Output folder:       {Path(OUTPUT_DIR).resolve()}")

    if missing_labels:
        print(f"\n[NOTE] {len(missing_labels)} images had no matching label file, e.g.:")
        for p in missing_labels[:5]:
            print(f"    - {p}")

    sanity_check(out_images, out_labels)


def sanity_check(images_dir: Path, labels_dir: Path):
    """
    Reports images with no label, labels that are empty, and labels with
    no matching image. Does NOT check for duplicate images yet.
    """
    image_stems = {p.stem for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
    label_files = {p.stem: p for p in labels_dir.iterdir() if p.is_file()}

    missing_label = sorted(image_stems - label_files.keys())
    orphaned_label = sorted(label_files.keys() - image_stems)

    empty_label = []
    for stem, path in label_files.items():
        if stem in image_stems and path.stat().st_size == 0:
            empty_label.append(stem)
        elif stem in image_stems and path.suffix == ".txt":
            content = path.read_text().strip()
            if not content:
                empty_label.append(stem)

    print("\n--- Sanity check ---")
    print(f"  Images total:               {len(image_stems)}")
    print(f"  Labels total:               {len(label_files)}")
    print(f"  Images with NO label file:  {len(missing_label)}")
    print(f"  Labels with EMPTY content:  {len(empty_label)}")
    print(f"  Orphaned labels (no image): {len(orphaned_label)}")

    if missing_label:
        print("\n  Examples of images missing labels:")
        for s in missing_label[:5]:
            print(f"    - {s}")
    if empty_label:
        print("\n  Examples of empty labels:")
        for s in empty_label[:5]:
            print(f"    - {s}")
    if orphaned_label:
        print("\n  Examples of orphaned labels:")
        for s in orphaned_label[:5]:
            print(f"    - {s}")


if __name__ == "__main__":
    merge()

### `merge_unknown.py`

merge_unknown_into_rat.py

In [ ]:
"""
merge_unknown_into_rat.py

Now that you've manually reviewed the unknown_0 crops and confirmed they're
all actually rats, this rewrites every label file in the merged dataset:
every box with class ID 1 (unknown_0) becomes class ID 0 (rat).

After this runs, the merged dataset has a single class: "rat" (ID 0).
It also rewrites data.yaml to reflect that.

Usage:
    python merge_unknown_into_rat.py
"""

import sys
from pathlib import Path
import yaml

MERGED_DIR = "datasets/merged_dataset"

OLD_UNKNOWN_ID = 1   # "unknown_0" in the current merged data.yaml
NEW_RAT_ID = 0        # "rat"

FINAL_CLASSES = ["rat"]


def main():
    labels_dir = Path(MERGED_DIR) / "labels"
    label_files = sorted(p for p in labels_dir.iterdir() if p.suffix.lower() == ".txt")
    total = len(label_files)

    print(f"Found {total} label files to check.\n")

    files_changed = 0
    boxes_changed = 0

    for scanned, label_path in enumerate(label_files, start=1):
        lines = label_path.read_text().splitlines()
        new_lines = []
        file_touched = False

        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            if class_id == OLD_UNKNOWN_ID:
                parts[0] = str(NEW_RAT_ID)
                file_touched = True
                boxes_changed += 1
            new_lines.append(" ".join(parts))

        if file_touched:
            label_path.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))
            files_changed += 1

        # Live progress line, overwritten in place (no scroll spam).
        pct = (scanned / total * 100) if total else 100
        sys.stdout.write(
            f"\r  checked {scanned}/{total} ({pct:.1f}%) "
            f"| files changed so far: {files_changed} | boxes changed so far: {boxes_changed}"
        )
        sys.stdout.flush()

    print()  # newline after the in-place progress line

    # Update data.yaml to reflect the single final class
    yaml_path = Path(MERGED_DIR) / "data.yaml"
    merged_yaml = {"nc": len(FINAL_CLASSES), "names": FINAL_CLASSES}
    with open(yaml_path, "w") as f:
        yaml.safe_dump(merged_yaml, f)

    print("Done.")
    print(f"  Label files changed: {files_changed} / {total}")
    print(f"  Boxes reclassified:  {boxes_changed}")
    print(f"  data.yaml updated to a single class: {FINAL_CLASSES}")
    print(f"  Output folder: {Path(MERGED_DIR).resolve()}")


if __name__ == "__main__":
    main()

### `remove.py`

remove_runs_images_from_merged.py

In [ ]:
"""
remove_runs_images_from_merged.py

You have a folder called "runs" full of images. This script:
  1. Lists every image found in runs/ (recursively, in case they're in subfolders).
  2. For each one, finds the image with the same filename in merged_dataset/images
     and removes it (along with its matching label in merged_dataset/labels, if any).

Nothing is deleted forever by default — matched files are moved to a
"removed_from_merged" backup folder instead, so you can double check before
permanently deleting them. Set PERMANENTLY_DELETE = True below if you're sure
and just want them gone.

Usage:
    python remove_runs_images_from_merged.py
"""

import shutil
import sys
from pathlib import Path

RUNS_DIR = "datasets/runs"
MERGED_DIR = "datasets/merged_dataset"
BACKUP_DIR = "datasets/removed_from_merged"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# If True, matched files are permanently deleted instead of moved to BACKUP_DIR.
PERMANENTLY_DELETE = False


def main():
    runs_dir = Path(RUNS_DIR)
    merged_images_dir = Path(MERGED_DIR) / "images"
    merged_labels_dir = Path(MERGED_DIR) / "labels"

    if not runs_dir.is_dir():
        print(f"ERROR: runs folder not found: {runs_dir.resolve()}")
        sys.exit(1)
    if not merged_images_dir.is_dir():
        print(f"ERROR: merged_dataset/images not found: {merged_images_dir.resolve()}")
        sys.exit(1)

    # Find every image in runs/ (recursive, in case they're nested in subfolders)
    runs_images = sorted(
        p for p in runs_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS
    )

    print(f"Found {len(runs_images)} image(s) in '{runs_dir}':\n")
    for p in runs_images:
        print(f"  {p.relative_to(runs_dir)}")
    print()

    if not runs_images:
        print("Nothing to do.")
        return

    if not PERMANENTLY_DELETE:
        backup_images_dir = Path(BACKUP_DIR) / "images"
        backup_labels_dir = Path(BACKUP_DIR) / "labels"
        backup_images_dir.mkdir(parents=True, exist_ok=True)
        backup_labels_dir.mkdir(parents=True, exist_ok=True)

    removed = 0
    not_found = []

    for img in runs_images:
        match = merged_images_dir / img.name
        if not match.exists():
            not_found.append(img.name)
            continue

        label_match = merged_labels_dir / f"{match.stem}.txt"

        if PERMANENTLY_DELETE:
            match.unlink()
            if label_match.exists():
                label_match.unlink()
        else:
            shutil.move(str(match), str(backup_images_dir / match.name))
            if label_match.exists():
                shutil.move(str(label_match), str(backup_labels_dir / label_match.name))

        removed += 1

    print("Done.")
    print(f"  Images removed from merged_dataset: {removed}")
    if not PERMANENTLY_DELETE:
        print(f"  (moved to backup folder: {Path(BACKUP_DIR).resolve()})")
    if not_found:
        print(f"  {len(not_found)} image(s) from 'runs' had no match in merged_dataset:")
        for name in not_found:
            print(f"    - {name}")


if __name__ == "__main__":
    main()

### `remove_duplicates.py`

dedupe_images_ssim.py

In [ ]:
"""
dedupe_images_ssim.py

Compares the FULL, UNCROPPED image (not just the labeled box region) to
find duplicate / near-duplicate images in the merged dataset. Confirmed
duplicates (image + label) are moved to a review folder -- nothing is
deleted.

WHY TWO STAGES:
Running SSIM on every possible PAIR of images is O(n^2) and each individual
comparison is fairly expensive -- on a few thousand images that's millions
of SSIM calls, which is why the plain SSIM-only version of this script was
slow. This version fixes that with a two-stage pipeline:

    1. Load images
    2. Compute a pHash (perceptual hash) for every image -- cheap, fast
    3. Group images with SIMILAR hashes (loose threshold, high recall --
       this is just a candidate filter, not the final decision)
    4. Run SSIM only INSIDE each candidate group, to confirm which of the
       candidates are actually true duplicates (strict threshold, high
       precision)

pHash alone can be fooled by genuinely different images that happen to
hash similarly (rare, but possible), so SSIM still makes the final call --
it's just no longer being run on every pair in the whole dataset, only on
the much smaller set of images that already look close by hash. This
keeps the expensive comparison scoped to where it's actually needed.

HASH_GROUP_THRESHOLD = 10 (Hamming distance, out of 64 bits): deliberately
loose. Its only job is to make sure real duplicates aren't missed before
SSIM gets a chance to look at them -- a few unrelated images ending up in
the same candidate group is fine and cheap, since SSIM filters those back
out in the next step.

CONTENT_THRESHOLD = 0.99 (SSIM, -1 to 1 range, 1.0 = structurally
identical): this is the strict, final decision. Only images that are
essentially the same picture will be confirmed as duplicates. Lower it
slightly (e.g. 0.95) if real duplicates are being missed at the SSIM stage.

SAFETY: nothing is deleted. Duplicates are MOVED to a review folder so you
can double check before permanently removing them.

Usage:
    pip install pillow numpy scikit-image imagehash --break-system-packages
    python dedupe_images_ssim.py
"""

import shutil
import sys
from pathlib import Path

import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import imagehash

MERGED_DIR = "datasets/merged_dataset"
REVIEW_DIR = "datasets/duplicates_review_ssim_merged"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# Stage 1 (pHash) -- loose candidate filter. See explanation above.
HASH_SIZE = 8  # 8x8 = 64-bit hash
HASH_GROUP_THRESHOLD = 10

# Stage 2 (SSIM) -- strict final confirmation. See explanation above.
CONTENT_THRESHOLD = 0.99

# Every image is resized to this fixed size before SSIM comparison, so SSIM
# can compare them (needs matching shapes) and comparison stays fast.
COMPARE_SIZE = (128, 128)


def build_records(images_dir: Path):
    """
    Load every image once, computing BOTH its pHash (for the fast stage-1
    candidate filter) and its resized grayscale array (for the stage-2 SSIM
    confirmation, done later only within candidate groups). Progress shown
    since this touches every file.
    """
    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(image_paths)
    print(f"Loading {total} images and computing pHashes...\n")

    hashes = {}
    grays = {}
    for i, img_path in enumerate(image_paths, start=1):
        try:
            with Image.open(img_path) as im:
                im_rgb = im.convert("RGB")
                hashes[img_path] = int(str(imagehash.phash(im_rgb, hash_size=HASH_SIZE)), 16)
                gray = im_rgb.convert("L").resize(COMPARE_SIZE, Image.LANCZOS)
                grays[img_path] = np.array(gray, dtype=np.uint8)
        except Exception as e:
            print(f"\n[WARNING] Could not load {img_path.name}: {e}")

        pct = i / total * 100 if total else 100
        sys.stdout.write(f"\r  loaded {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line
    return hashes, grays


def group_by_hash(hashes: dict, threshold: int):
    """
    Stage 1: cheap candidate filter. Groups images whose pHash is within
    `threshold` Hamming-distance bits of each other. Vectorized with numpy
    in row blocks so it stays fast even on a few thousand images. This is
    deliberately loose (high recall) -- SSIM makes the final call afterward.
    """
    paths = list(hashes.keys())
    n = len(paths)
    hash_array = np.array([hashes[p] for p in paths], dtype=np.uint64)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    print(f"\nGrouping {n} images by similar pHash (candidate threshold={threshold})...\n")

    block_size = 500
    for start in range(0, n, block_size):
        end = min(start + block_size, n)
        block = hash_array[start:end]
        xor = block[:, None] ^ hash_array[None, :]
        dist = np.unpackbits(xor.view(np.uint8), axis=-1).reshape(xor.shape[0], xor.shape[1], -1).sum(axis=-1)

        for local_i in range(end - start):
            global_i = start + local_i
            close = np.where(dist[local_i] <= threshold)[0]
            for j in close:
                if j != global_i:
                    union(global_i, int(j))

        pct = end / n * 100 if n else 100
        sys.stdout.write(f"\r  grouped {end}/{n} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line

    groups = {}
    for i in range(n):
        root = find(i)
        groups.setdefault(root, []).append(paths[i])

    return [g for g in groups.values() if len(g) > 1]


def crops_match(image_a: np.ndarray, image_b: np.ndarray) -> bool:
    """
    Compare two (whole, uncropped) images using SSIM. If the score is below
    CONTENT_THRESHOLD, they're NOT considered the same content.
    """
    score = ssim(image_a, image_b)
    if score < CONTENT_THRESHOLD:
        return False
    return True


def confirm_with_ssim(candidate_groups, grays: dict):
    """
    Stage 2: within each pHash candidate group (usually small), run pairwise
    SSIM to confirm true duplicates. A candidate group can split into
    multiple confirmed groups, or shrink if some members turn out not to
    really match. Returns the final list of confirmed duplicate groups.
    """
    total_candidates = sum(len(g) for g in candidate_groups)
    print(f"\nConfirming {len(candidate_groups)} candidate group(s) "
          f"({total_candidates} images total) with SSIM (threshold={CONTENT_THRESHOLD})...\n")

    confirmed_groups = []
    checked = 0
    for group in candidate_groups:
        parent = {p: p for p in group}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb

        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                p1, p2 = group[i], group[j]
                if crops_match(grays[p1], grays[p2]):
                    union(p1, p2)

        sub_groups = {}
        for p in group:
            root = find(p)
            sub_groups.setdefault(root, []).append(p)

        confirmed_groups.extend(g for g in sub_groups.values() if len(g) > 1)

        checked += len(group)
        pct = checked / total_candidates * 100 if total_candidates else 100
        sys.stdout.write(f"\r  confirmed {checked}/{total_candidates} candidate images ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line
    return confirmed_groups


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    out_dir = Path(REVIEW_DIR)
    out_images = out_dir / "images"
    out_labels = out_dir / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    hashes, grays = build_records(images_dir)

    if len(hashes) < 2:
        print("\nNot enough images to compare.")
        return

    candidate_groups = group_by_hash(hashes, HASH_GROUP_THRESHOLD)
    print(f"\nFound {len(candidate_groups)} candidate group(s) from pHash pre-filter.")

    if not candidate_groups:
        groups = []
    else:
        groups = confirm_with_ssim(candidate_groups, grays)

    print(f"\nFound {len(groups)} duplicate group(s).\n")

    moved_images = 0
    for group_num, group in enumerate(groups, start=1):
        group_sorted = sorted(group, key=lambda p: p.name)
        keeper = group_sorted[0]
        losers = group_sorted[1:]

        print(f"  [group {group_num}] keeping {keeper.name}, moving {len(losers)} duplicate(s): "
              f"{', '.join(p.name for p in losers)}")

        for img_path in losers:
            dst_img = out_images / img_path.name
            shutil.move(str(img_path), str(dst_img))

            label_path = labels_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.move(str(label_path), str(out_labels / label_path.name))

            moved_images += 1

    remaining_images = sum(1 for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

    print("\nDone.")
    print(f"  Duplicate groups found:     {len(groups)}")
    print(f"  Images removed (moved out): {moved_images}")
    print(f"  Images remaining in merged: {remaining_images}")
    print(f"  Review folder:              {out_dir.resolve()}")
    print("\nNothing was deleted. Check the review folder -- if the moved images")
    print("really are unwanted duplicates, delete that folder. If any group was")
    print("wrongly matched, move those specific files back into")
    print("merged_dataset/images and .../labels.")


if __name__ == "__main__":
    main()

### `renaming.py`

rename_dataset_sequential.py

In [ ]:
"""
rename_dataset_sequential.py

Renames every image (and its matching label file) in the merged dataset to
a sequential zero-padded number, e.g.:

    images/000001.jpg   labels/000001.txt
    images/000002.jpg   labels/000002.txt
    images/000003.jpg   labels/000003.txt
    ...

Files are processed in sorted (alphabetical) order of their current name,
so the numbering is stable and repeatable.

Renaming is done in TWO passes (first to temporary names, then to final
names) so it's safe even if some of the target numeric names happen to
already exist -- nothing gets overwritten or lost mid-run.

Usage:
    python rename_dataset_sequential.py
"""

import sys
from pathlib import Path

MERGED_DIR = "datasets/merged_dataset"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# Number of digits to zero-pad to, e.g. 6 -> 000001.jpg. Make sure this is
# large enough for your total image count (6 digits = up to 999,999).
PAD_WIDTH = 6


def main():
    images_dir = Path(MERGED_DIR) / "images"
    labels_dir = Path(MERGED_DIR) / "labels"

    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(image_paths)
    print(f"Found {total} images to rename.\n")

    if total == 0:
        print("Nothing to do.")
        return

    # --- Pass 1: rename everything to temporary names -------------------
    # Avoids collisions if a target numeric name (e.g. "000042.jpg")
    # happens to already exist among the current filenames.
    print("Pass 1/2: renaming to temporary names...\n")
    temp_pairs = []  # (final_stem, temp_img_path, temp_label_path_or_None)

    for i, img_path in enumerate(image_paths, start=1):
        final_stem = str(i).zfill(PAD_WIDTH)
        temp_img_path = images_dir / f"__tmp_{i}{img_path.suffix}"
        img_path.rename(temp_img_path)

        label_path = labels_dir / f"{img_path.stem}.txt"
        temp_label_path = None
        if label_path.exists():
            temp_label_path = labels_dir / f"__tmp_{i}.txt"
            label_path.rename(temp_label_path)

        temp_pairs.append((final_stem, temp_img_path, temp_label_path))

        pct = i / total * 100
        sys.stdout.write(f"\r  renamed {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line

    # --- Pass 2: rename from temporary names to final numbered names ----
    print("\nPass 2/2: renaming to final numbered names...\n")
    missing_labels = 0

    for i, (final_stem, temp_img_path, temp_label_path) in enumerate(temp_pairs, start=1):
        final_img_path = images_dir / f"{final_stem}{temp_img_path.suffix}"
        temp_img_path.rename(final_img_path)

        if temp_label_path is not None:
            final_label_path = labels_dir / f"{final_stem}.txt"
            temp_label_path.rename(final_label_path)
        else:
            missing_labels += 1

        pct = i / total * 100
        sys.stdout.write(f"\r  renamed {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

    print()  # newline after progress line

    print("\nDone.")
    print(f"  Images renamed:     {total}")
    print(f"  Naming scheme:      {'0' * PAD_WIDTH}.ext  (e.g. {'1'.zfill(PAD_WIDTH)}.jpg)")
    if missing_labels:
        print(f"  [NOTE] {missing_labels} image(s) had no matching label file, so only the image was renamed.")


if __name__ == "__main__":
    main()

### `restore_labels.py`

restore_labeled_images.py

In [ ]:
"""
restore_labeled_images.py

After reviewing the "empty labels" images in CVAT, you exported YOLO-format
labels (obj_train_data/*.txt, obj.data, obj.names, train.txt). Some flagged
images were deleted during review, so they'll have no matching label file.

This script:
  1. Looks at every label file in the CVAT export (obj_train_data/).
  2. For each one, finds the matching image in the empty_labels_review folder.
  3. Moves that image + its new label into merged_dataset/images and
     merged_dataset/labels.
  4. Any image left in empty_labels_review with no matching CVAT label
     (i.e. you deleted it in CVAT) is simply left where it is.

Nothing is deleted by this script.

Usage:
    python restore_labeled_images.py
"""

import shutil
import sys
from pathlib import Path

MERGED_DIR = "datasets/merged_dataset"
EMPTY_LABELS_DIR = "datasets/empty_labels_review"
CVAT_EXPORT_DIR = "datasets/annotations/obj_train_data"  # folder containing the new .txt label files from CVAT

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def main():
    cvat_labels_dir = Path(CVAT_EXPORT_DIR)
    review_images_dir = Path(EMPTY_LABELS_DIR) / "images"
    review_labels_dir = Path(EMPTY_LABELS_DIR) / "labels"

    merged_images_dir = Path(MERGED_DIR) / "images"
    merged_labels_dir = Path(MERGED_DIR) / "labels"
    merged_images_dir.mkdir(parents=True, exist_ok=True)
    merged_labels_dir.mkdir(parents=True, exist_ok=True)

    if not cvat_labels_dir.is_dir():
        print(f"ERROR: CVAT export folder not found: {cvat_labels_dir.resolve()}")
        sys.exit(1)
    if not review_images_dir.is_dir():
        print(f"ERROR: review images folder not found: {review_images_dir.resolve()}")
        sys.exit(1)

    # Map of stem -> image path, for every image currently sitting in review
    review_images = {
        p.stem: p for p in review_images_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTS
    }

    # New label files from the CVAT export
    cvat_labels = sorted(cvat_labels_dir.glob("*.txt"))

    total = len(cvat_labels)
    print(f"Found {total} label files in CVAT export.")
    print(f"Found {len(review_images)} images waiting in review folder.\n")

    moved = 0
    missing_image_for_label = []

    for i, label_path in enumerate(cvat_labels, start=1):
        stem = label_path.stem
        img_path = review_images.get(stem)

        if img_path is None:
            # A label exists but we can't find the matching image in review.
            # (Shouldn't normally happen — flag it instead of silently skipping.)
            missing_image_for_label.append(label_path.name)
        else:
            # Move image -> merged_dataset/images
            shutil.move(str(img_path), str(merged_images_dir / img_path.name))
            # Move (new) label -> merged_dataset/labels
            shutil.move(str(label_path), str(merged_labels_dir / label_path.name))
            # Remove it from the lookup so we can compute leftovers afterward
            del review_images[stem]
            moved += 1

        pct = i / total * 100 if total else 100
        sys.stdout.write(f"\r  processed {i}/{total} ({pct:.1f}%) | moved so far: {moved}")
        sys.stdout.flush()

    print()  # newline after progress bar

    # Anything still left in review_images had no label -> was deleted in CVAT.
    # We leave those files untouched in EMPTY_LABELS_DIR/images.
    leftover = sorted(review_images.values())

    # Clean up any now-orphaned label files still sitting in review/labels
    # (old empty labels for images that got moved out)
    orphan_labels_removed = 0
    if review_labels_dir.is_dir():
        for lbl in review_labels_dir.iterdir():
            if lbl.suffix == ".txt" and lbl.stem not in {p.stem for p in leftover}:
                # this old empty label's image is gone (either moved to merged or deleted)
                # only remove it if there's truly no image left for it in review
                if lbl.stem not in review_images_dir_stems(review_images_dir):
                    lbl.unlink()
                    orphan_labels_removed += 1

    print("\nDone.")
    print(f"  Images + labels moved back to merged_dataset: {moved}")
    print(f"  Images left in review (no label -> deleted in CVAT): {len(leftover)}")
    if missing_image_for_label:
        print(f"  WARNING: {len(missing_image_for_label)} label file(s) had no matching "
              f"image in review folder:")
        for name in missing_image_for_label:
            print(f"    - {name}")
    print(f"  Old empty label leftovers cleaned up: {orphan_labels_removed}")
    print(f"\n  merged_dataset:      {Path(MERGED_DIR).resolve()}")
    print(f"  review folder (rest): {Path(EMPTY_LABELS_DIR).resolve()}")


def review_images_dir_stems(review_images_dir: Path):
    return {p.stem for p in review_images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}


if __name__ == "__main__":
    main()

### `sanity_check.py`

Check for mathematically impossible or bad bounding boxes

In [ ]:
from pathlib import Path

LABELS_DIR = "datasets/merged_dataset/labels"

def check_label_math(labels_path):
    bad_files = 0
    for txt_file in Path(labels_path).glob("*.txt"):
        with open(txt_file, 'r') as f:
            for line_num, line in enumerate(f, 1):
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                
                try:
                    _, x_c, y_c, w, h = map(float, parts)
                except ValueError:
                    continue

                # Check for mathematically impossible or bad bounding boxes
                # YOLO coords should be between 0 and 1. 
                # Width/Height should be > 0 and usually <= 1
                if not (0 <= x_c <= 1 and 0 <= y_c <= 1):
                    print(f"Bad center coord in {txt_file.name} (Line {line_num}): {line.strip()}")
                    bad_files += 1
                elif not (0 < w <= 1 and 0 < h <= 1):
                    print(f"Bad width/height in {txt_file.name} (Line {line_num}): {line.strip()}")
                    bad_files += 1

    if bad_files == 0:
        print("SUCCESS: All 11,000 labels have mathematically valid coordinates!")
    else:
        print(f"WARNING: Found {bad_files} lines with impossible coordinates.")

if __name__ == "__main__":
    check_label_math(LABELS_DIR)

### `split_data_hashed.py`

Configuration

In [ ]:
import os
import random
import shutil
from collections import defaultdict
from pathlib import Path
from PIL import Image
import imagehash

# Configuration
DATASET_DIR = Path("datasets/merged_dataset/ei_upload")
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
# How similar do backgrounds need to be to be considered the "same place"?
# Lower number = stricter (must look almost identical). Higher number = more loose.
HAMMING_THRESHOLD = 10 

def split_dataset_by_background():
    if not DATASET_DIR.exists():
        print(f"Error: Directory '{DATASET_DIR}' does not exist.")
        return

    print("Step 1: Creating visual fingerprints for all images (this may take a minute)...")
    valid_pairs = []
    for file_path in DATASET_DIR.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            json_path = file_path.with_suffix('.json')
            if json_path.exists():
                try:
                    # Create a fingerprint of the image
                    img_hash = imagehash.phash(Image.open(file_path))
                    valid_pairs.append((str(img_hash), file_path, json_path))
                except Exception as e:
                    print(f"Could not process {file_path.name}: {e}")

    if not valid_pairs:
        print("No valid image/JSON pairs found.")
        return

    # 2. Sort images by their visual fingerprint!
    # This puts images with the exact same background right next to each other.
    valid_pairs.sort(key=lambda x: x[0])

    print("Step 2: Grouping similar backgrounds together...")
    groups = []
    current_group = [valid_pairs[0]]
    
    for i in range(1, len(valid_pairs)):
        current_hash = imagehash.hex_to_hash(valid_pairs[i][0])
        prev_hash = imagehash.hex_to_hash(valid_pairs[i-1][0])
        
        # If the difference between fingerprints is small, they are the same place
        if (current_hash - prev_hash) < HAMMING_THRESHOLD:
            current_group.append(valid_pairs[i])
        else:
            # Background changed, save the current group and start a new one
            groups.append(current_group)
            current_group = [valid_pairs[i]]
    groups.append(current_group) # Add the final group

    print(f"Found {len(groups)} unique scene locations.")

    # 3. Shuffle the groups (not the individual images)
    random.seed(RANDOM_SEED)
    random.shuffle(groups)

    # 4. Calculate split sizes based on the number of unique scenes
    total_scenes = len(groups)
    train_scene_count = int(total_scenes * TRAIN_RATIO)
    
    train_groups = groups[:train_scene_count]
    test_groups = groups[train_scene_count:]

    # 5. Create directories
    train_dir = DATASET_DIR / "training"
    test_dir = DATASET_DIR / "testing"
    train_dir.mkdir(exist_ok=True)
    test_dir.mkdir(exist_ok=True)

    # 6. Move files
    train_count = 0
    test_count = 0

    print("Step 3: Moving files to training and testing folders...")
    for group in train_groups:
        for _, img, json_file in group:
            shutil.move(str(img), str(train_dir / img.name))
            shutil.move(str(json_file), str(train_dir / json_file.name))
            train_count += 1

    for group in test_groups:
        for _, img, json_file in group:
            shutil.move(str(img), str(test_dir / img.name))
            shutil.move(str(json_file), str(test_dir / json_file.name))
            test_count += 1

    print("\n✅ Dataset successfully split by background scene (No Data Leakage)!")
    print(f"Unique Training Scenes: {len(train_groups)} ({train_count} images)")
    print(f"Unique Testing Scenes:  {len(test_groups)} ({test_count} images)")

if __name__ == "__main__":
    split_dataset_by_background()